In [ ]:
import __main__
import sys, os
import time
project_root = os.path.abspath("..")  # adjust if notebook is elsewhere
sys.path.insert(0, project_root)
from typing import Dict, List, Literal, Tuple, Optional, Any, Union
import logging
import random
import umap
import yaml
from dataclasses import dataclass

import category_encoders as ce
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import numexpr as ne # makes numpy operations faster
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm

from scipy.signal import periodogram

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import NearestNeighbors, KernelDensity
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

from catboost import CatBoostRegressor, CatBoostClassifier
from pyriemann.tangentspace import TangentSpace
from ucimlrepo import fetch_ucirepo

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    # print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

from geo_utils import compute_cyclicity_score, compute_multicyclicity_scores, append_run_to_csv, \
split_dataset_to_linear_and_cyclic, scale_train_and_test_sets, drop_low_variance_cols, Windowing, Periodicity, Predictors
from geo_encoders import EuclidEncoder, SphericalEncoder, Decoder, LSTMEncoderEuclid, LSTMSphericalEncoder, \
LSTMToroidalEncoder, LSTMDecoder, MLPDecoder, Reparam, MixedEncoder, WithSplit, NoSplit, kl_gaussian, kl_vmf_uniform, \
regularization_vmf, early_stop, fit_catboost_multi, evaluate_model_full, estimate_entropy, pool_latents
from storage_dicts import dataset_attributes
import src.param_config.config_paths as P
from src.utils.io_utils import read_yaml_params
import data_processor as dp
from topolin import Sphlin#, #Oldtor, Torlin, TopolinPlots

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
"[✅ RUN] Params"

@dataclass
class RunParams:
    """Data class for run parameters"""
    window_size: int
    z_dim_total: int
    hidden_dim: int
    lr_optimizer: float
    batch_size: int
    epochs: int
    horizon: int
    cyclic_threshold: float
    earlystop_patience: int = 8
    lambda_recon: float = 1.0
    # lambda_latent: float = 1e-3
    lambda_kl_euc: float = 1e-3
    lambda_kl_sph: float = 1e-3
    lambda_pred: float = 1.0
    distinct_periods: int = 1
    task: str = "forecast"  # forecast, nowcast, tabular

    def __repr__(self) -> str:
        return ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in vars(self).items())

try:
    geo_params_path = "geo_params.yaml"
    geo_params      = read_yaml_params(geo_params_path)
    dataset         = geo_params["dataset"]["general"]["desired_dataset"]
    p_dict          = geo_params["dataset"][dataset]["override"]
    print("☑️ Using YAML override params.")
except KeyError: # use your manual dict
    p_dict          = geo_params["dataset"]["default"]
    print("❌ Override not found. Using manual default_params.")

p = RunParams(**p_dict)

earlystop_patience = geo_params["dataset"]["general"].get("earlystop_patience", 8)
epochs             = geo_params["dataset"]["general"].get("epochs", 50)
print(f"📊 {dataset=}")

file_loc       = dp.dataset_dict[dataset]["file_loc"]
y_cols         = dp.dataset_dict[dataset]["y_cols"]
X_orig, y_orig = dp.dataset_dict[dataset]["function"](file_loc, y_cols)

if "processing" in dp.dataset_dict[dataset]:
    X_orig, y_orig = dp.dataset_dict[dataset]["processing"](X_orig, y_orig)

prediction_task = "forecast"

try:
    sliding_window_frac = dataset_attributes[dataset]["sliding_window_frac"] #sliding_window_frac_dict[dataset]
    print(f"🪟 Using sliding_window_frac from dict: {sliding_window_frac}")
except:
    sliding_window_frac = 0.25 # 10-25% of window length

sliding_size = int(p.window_size * sliding_window_frac)
if prediction_task == "tabular":
    window_size  = 1
    sliding_size = 1
try:
    window_size = p.window_size
except:
    p.window_size = window_size

p.horizon = 1
print(f"🧩 {p.window_size=}, {sliding_size=}, {p.horizon=}")

# for split scenario
# hidden_dim_split= int(hidden_dim / np.sqrt(2)) # approximation of (§2.1 of https://arxiv.org/pdf/2001.08361)
p.cyclic_threshold     = dataset_attributes[dataset]["periodic_threshold"]
angular_col_names_list = dataset_attributes[dataset]["angular_features"]

X = X_orig.copy()
y = y_orig.copy()
X.drop(columns= dataset_attributes[dataset]["cols_to_drop"], inplace=True, errors='ignore') # drop non-numeric if present
X = drop_low_variance_cols(X) #drop low var cols


for col in range(X.shape[1]):
    data_col        = X.iloc[:, col].to_numpy().flatten()
    cyclicity_score = compute_cyclicity_score(data_col)
    print(f"col {col} cycl. {cyclicity_score:.4f}")

# 1. train/test split without shuffling (time series)
train_test_split_ratio = 0.2
if prediction_task == "tabular": # shuffle tabular
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=train_test_split_ratio, random_state=42, shuffle=True)
else: # dont shuffle timeseries
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=train_test_split_ratio, random_state=42, shuffle=False) #timeseries order matters

test_set_size = train_test_split_ratio * X.shape[0]
if test_set_size < p.window_size: # test set < window size
    print(f"⚠️ Test set {test_set_size} < window size {p.window_size}")
    X_train, X_test, y_train, y_test = Windowing.split_timeseries_by_windows(X=X, y=y, window_size=p.window_size, 
                                        sliding_fraction=sliding_window_frac, test_ratio=train_test_split_ratio,)

# # =========
# add noise to x
# rows, cols = X.shape
# noise      = 2 * X.values * torch.randn(rows, cols).numpy()
# X          = X + noise

def add_phase_jitter(X, max_shift=2):
    """
    Randomly shifts rows up or down slightly to simulate 
    timing unsynchronization in sensors.
    """
    X_jittered = X.copy()
    rows, cols = X.shape
    for j in range(cols):
        # Generate a random integer shift for this feature
        shift = np.random.randint(-max_shift, max_shift + 1)
        if shift != 0:
            X_jittered.iloc[:, j] = np.roll(X.iloc[:, j], shift)
    return X_jittered

# X = add_phase_jitter(X, max_shift=10) # 3-step jitter

noise_frac = .5
X_train = X_train + (noise_frac * X_train.std().values * np.random.randn(*X_train.shape))
X_test  = X_test  + (noise_frac * X_test.std().values  * np.random.randn(*X_test.shape))
# # =========


print(f"shape of X: {X_orig.shape}, y: {y_orig.shape}")
print(f"[Fixed] X_train: {X_train.shape}, X_test: {X_test.shape}, ratio: {len(X_test)/(len(X_train)+len(X_test)):.2f}")


In [ ]:
"🍩🍝 new torlin"
from geo_encoders import MLPPredHead

class JointLSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, z_euc_dim, num_periods, n_layers: int =2):
        super().__init__()
        self.num_periods = num_periods
        self.lstm        = nn.LSTM(input_dim, hidden_dim, num_layers=n_layers, batch_first=True)
        self.head_euc    = nn.Linear(hidden_dim, z_euc_dim * 2)
        self.head_torus_mu    = nn.Linear(hidden_dim, num_periods * 2)
        self.head_torus_kappa = nn.Linear(hidden_dim, num_periods)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = h[-1]
        euc_params     = self.head_euc(h)
        mu_e, logvar_e = torch.chunk(euc_params, 2, dim=-1)
        mu_s_raw = self.head_torus_mu(h).view(-1, self.num_periods, 2)
        mu_s     = F.normalize(mu_s_raw, p=2, dim=-1) 
        logkappa = self.head_torus_kappa(h).unsqueeze(-1) 
        return mu_e, logvar_e, mu_s, logkappa

class Torlin(Sphlin):
    @staticmethod
    def train_step(x_all, y_win, encoder, decoder, pred_head, lambdas, epoch, pred_mode="ssl"):
        mu_e, logvar_e, mu_s, logkappa = encoder(x_all)

        z_e = Torlin.sample_gaussian(mu_e, logvar_e)
        kl_e = Torlin.kl_gaussian(mu_e, logvar_e)

        z_s_list, kl_s_total = [], 0
        for i in range(encoder.num_periods):
            z_c = Torlin.sample_vmf(mu_s[:, i, :], logkappa[:, i, :])
            z_s_list.append(z_c)
            kl_s_total += Torlin.kl_vmf(mu_s[:, i, :], logkappa[:, i, :])
        kl_s_total /= encoder.num_periods

        z_s = torch.cat(z_s_list, dim=-1)
        z_total = torch.cat([z_e, z_s], dim=-1)

        x_target = x_all.reshape(x_all.size(0), -1)
        x_hat = decoder(z_total).reshape(x_target.shape)
        recon_loss = F.mse_loss(x_hat, x_target)

        # ---- USE pred_mode HERE ----
        z_pred = z_e if pred_mode == "ssl" else z_total
        y_hat = pred_head(z_pred)

        y_win = y_win.unsqueeze(-1) if y_win.dim() == 1 else y_win
        mask = ~torch.isnan(y_win)
        pred_loss = F.mse_loss(y_hat[mask], y_win[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=y_win.device)

        label_frac = mask.float().mean()
        pred_weight = lambdas["pred"] * label_frac
        kl_weight = 0.1 if epoch < 20 else 0.5 if epoch < 60 else 1.0

        total_loss = (lambdas["reconstr"] * recon_loss +
                      kl_weight * (lambdas["euc"] * kl_e + lambdas["sph"] * kl_s_total) +
                      pred_weight * pred_loss)

        return {"total": total_loss, "recon": recon_loss, "kl_e": kl_e, "kl_s": kl_s_total, "pred": pred_loss}

    @staticmethod
    def train_epoch(loader, encoder, decoder, pred_head, optimizer, lambdas, device, epoch, pred_mode="ssl"):
        encoder.train(); decoder.train(); pred_head.train()
        totals, count = {}, 0
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            losses = Torlin.train_step(x_batch, y_batch, encoder, decoder, pred_head, lambdas, epoch, pred_mode=pred_mode)
            losses["total"].backward()
            optimizer.step()
            for k, v in losses.items():
                totals[k] = totals.get(k, 0.0) + v.item()
            count += 1
        return {k: v / count for k, v in totals.items()}

    @staticmethod
    def encode_full_dataset(X_w, encoder, device, batch_size=64):
        encoder.eval()
        zs = []
        with torch.no_grad():
            for i in range(0, len(X_w), batch_size):
                batch = X_w[i:i+batch_size].to(device)
                mu_e, _, mu_s, _ = encoder(batch)
                z_torus = mu_s.reshape(mu_s.size(0), -1)
                zs.append(torch.cat([mu_e, z_torus], dim=-1).cpu())
        return torch.cat(zs, dim=0)

    @staticmethod
    def run_torlin(X_train, X_test, y_train, y_test, p, num_periods=5, sliding_size=10, pred_mode: Literal["ssl", "downstream"] = "ssl"):
        X_tr_s, X_te_s = scale_train_and_test_sets(X_train, X_test)
        y_tr_s, y_te_s = scale_train_and_test_sets(y_train, y_test)

        X_tr_w = Windowing.make_windows_from_X(X_tr_s, p.window_size, sliding_size).to(device)
        X_te_w = Windowing.make_windows_from_X(X_te_s, p.window_size, sliding_size).to(device)
        y_tr_w = Windowing.make_windows_from_y(y_tr_s, p.window_size, sliding_size, task=p.task).reshape(-1, 1)
        y_te_w = Windowing.make_windows_from_y(y_te_s, p.window_size, sliding_size, task=p.task).reshape(-1, 1)

        min_len = min(X_tr_w.size(0), y_tr_w.shape[0])
        X_tr_w, y_tr_w = X_tr_w[:min_len], y_tr_w[:min_len]
        X_te_min = min(X_te_w.size(0), y_te_w.shape[0])
        X_te_w, y_te_w = X_te_w[:X_te_min], y_te_w[:X_te_min]

        loader = DataLoader(TensorDataset(X_tr_w, torch.tensor(y_tr_w, dtype=torch.float32)), batch_size=p.batch_size, shuffle=True)

        z_sph_dim = num_periods * 2
        z_euc_dim = max(4, p.z_dim_total - z_sph_dim)
        z_final_total = z_euc_dim + z_sph_dim

        enc = JointLSTMEncoder(X_train.shape[1], p.hidden_dim, z_euc_dim, num_periods, n_layers=2).to(device)
        dec = MLPDecoder(z_final_total, p.window_size, X_train.shape[1], p.hidden_dim).to(device)

        # ---- pred head depends on pred_mode ----
        pred_in_dim = z_euc_dim if pred_mode == "ssl" else z_final_total
        pred = MLPPredHead(pred_in_dim, 1, hidden_dim=p.hidden_dim*2).to(device)

        opt = torch.optim.AdamW(list(enc.parameters())+list(dec.parameters())+list(pred.parameters()), lr=p.lr_optimizer)
        lambdas = {"reconstr": p.lambda_recon, "pred": p.lambda_pred, "euc": p.lambda_kl_euc, "sph": p.lambda_kl_sph}

        for epoch in range(p.epochs):
            res = Torlin.train_epoch(loader, enc, dec, pred, opt, lambdas, device, epoch, pred_mode=pred_mode)
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss {res['total']:.2f} | Recon {res['recon']:.4f} | Pred {res['pred']:.4f} | KL_E {res['kl_e']:.4f} | KL_S {res['kl_s']:.4f}")

        Z_train = Torlin.encode_full_dataset(X_tr_w, enc, device)
        Z_test  = Torlin.encode_full_dataset(X_te_w, enc, device)

        pred.eval()
        with torch.no_grad():
            # y_hat = pred(Z_test.to(device)).cpu().numpy()
            z_pred_test = Z_test[:, :z_euc_dim] if pred_mode=="ssl" else Z_test
            y_hat       = pred(z_pred_test.to(device)).cpu().numpy()

        return root_mean_squared_error(y_te_w, y_hat), r2_score(y_te_w, y_hat), mean_absolute_error(y_te_w, y_hat), \
               (Z_train, Z_test), (y_hat, y_tr_w, y_te_w), (z_euc_dim, z_sph_dim), {}, (enc, pred, X_te_w)


rmse_torlin, r2_torlin, mae_torlin = [], [], []
num_distinct_periods = p.distinct_periods

for i in range(2):
    rmse_i, r2_i, mae_i, (Z_train, Z_test), (y_hat, y_train_w, y_test_w), (z_dim_euclid, z_dim_spher), _, (enc, pred, X_te_w) = \
        Torlin.run_torlin(X_train, X_test, y_train, y_test, p, num_periods=num_distinct_periods, sliding_size=sliding_size, pred_mode="downstream")
    rmse_torlin.append(rmse_i)
    r2_torlin.append(r2_i)
    mae_torlin.append(mae_i)
    print(f"Run {i+1} Results -> RMSE: {rmse_i:.4f}, R2: {r2_i:.4f}")

rmse_torlin_mean, rmse_torlin_std = np.mean(rmse_torlin), np.std(rmse_torlin)
mae_torlin_mean, mae_torlin_std   = np.mean(mae_torlin), np.std(mae_torlin)
r2_torlin_mean, r2_torlin_std     = np.mean(r2_torlin), np.std(r2_torlin)

print(f"  🍩🐢 Torlin RMSE={rmse_torlin_mean:.3f}±{rmse_torlin_std:.3f} \
        MAE={mae_torlin_mean:.3f}±{mae_torlin_std:.3f} \
        R2={r2_torlin_mean:.3f}±{r2_torlin_std:.3f}")

metrics = {"rmse": rmse_torlin_mean, "r2": r2_torlin_mean, "mae": mae_torlin_mean,}
metrics = {k: round(float(v), 4) for k, v in metrics.items()}
append_run_to_csv(params=p.__dict__, metrics=metrics, dataset=dataset, file_path="geo_results.csv", method=f"torlin",)


In [ ]:
"[🌐 Sphlin]: cyclic + linear split"

p.cyclic_threshold = .9499459491200945

rmse_sphlin, r2_sphlin, mae_sphlin = [], [], []
for i in range(2):
    rmse_i, r2_i, mae_i, (Z_train, Z_test), (y_hat, y_train_w, y_test_w), (z_dim_euclid, z_dim_spher), logs, \
        (enc_e, enc_s, pred_head, X_lin_te_w, X_cyc_te_w) = Sphlin.run_sphlin_LSTM(X_train, X_test, y_train, y_test, \
                                            p, sliding_size=sliding_size)#, manually_set_cols=angular_col_names_list)

    rmse_sphlin.append(rmse_i)
    r2_sphlin.append(r2_i)
    mae_sphlin.append(mae_i)
    print(f"Run {i+1} Results -> RMSE: {rmse_i:.4f}, R2: {r2_i:.4f}")

rmse_sphlin_mean, rmse_sphlin_std = np.mean(rmse_sphlin), np.std(rmse_sphlin)
mae_sphlin_mean, mae_sphlin_std   = np.mean(mae_sphlin), np.std(mae_sphlin)
r2_sphlin_mean, r2_sphlin_std     = np.mean(r2_sphlin), np.std(r2_sphlin)

print(f"  🔑 Sphlin RMSE={rmse_sphlin_mean:.3f}±{rmse_sphlin_std:.3f} \
        MAE={mae_sphlin_mean:.3f}±{mae_sphlin_std:.3f} \
        R2={r2_sphlin_mean:.3f}±{r2_sphlin_std:.3f}")
rmse_random, r2_random = Predictors.make_random_latent_prediction(Z_train, y_train, p, sliding_size=sliding_size, n_samples=Z_test.shape[0])
print(f"🫟 random latent pred. RMSE={rmse_random:.4f}, R2={r2_random:.4f}")

metrics = {"rmse": rmse_sphlin_mean, "r2": r2_sphlin_mean, "mae": mae_sphlin_mean,}
metrics = {k: round(float(v), 4) for k, v in metrics.items()}
append_run_to_csv(params=p.__dict__, metrics=metrics, dataset=dataset, file_path="geo_results.csv", method=f"sphlin_{p.cyclic_threshold:.2f}",)

cos_sim     = Predictors.cosine_similarity_samples(Z_train)
angular_var = Predictors.angular_variance_over_time(Z_train)
print(f"🎼 z_train: cos sim (↑) ={cos_sim:.4f}, angular var over time (↓): {angular_var:.4f}")

"🧠 sanity check: mean should not do better than us"
y_mean    = np.mean(y_train_w, axis=0)
rmse_mean = np.sqrt(mean_squared_error(y_train_w, np.tile(y_mean, (len(y_train_w), 1))))
print(f"🧠 Mean predictor RMSE: {rmse_mean:.4f}")

# rmse_pca, r2_pca = Predictors.make_latent_pca_prediction(Z_train, Z_test, y_train, y_test, p, n_components=0.95)
# print(f"PCA latent pred. RMSE: {rmse_pca:.4f}, R2: {r2_pca:.4f}")

# rmse_umap_cb, r2_umap_cb = Predictors.make_latent_umap_catboost(Z_train, Z_test, y_train, y_test, p,
#                                                      n_components=5, n_neighbors=15, min_dist=0.1)
# print(f"UMAP + CatBoost RMSE: {rmse_umap_cb:.4f}, R2: {r2_umap_cb:.4f}")

# y_hat, rmse, r2 = Predictors.run_riemann_catboost(Z_train, Z_test, y_train, y_test, window_size, sliding_size, prediction_task)
# print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
# rmse_i, r2_i, mae_i, (Z_train, Z_test), (y_hat, y_train_w, y_test_w), (z_dim_euclid, z_dim_spher), _, (enc, pred, X_te_w) = \
#     Torlin.run_torlin(X_train, X_test, y_train, y_test, p, num_periods=num_distinct_periods, sliding_size=sliding_size)

plt.plot(y_test_w, label="True")
plt.plot(y_hat, label="Pred")


In [ ]:
"""multistep FORECAST"""
from tqdm.auto import tqdm

def evaluate_recursive_forecast(enc, pred, X_window, y_ground_truth, p, steps=250, device="cuda"):
    enc.eval(); pred.eval()
    curr_w = X_window.clone().to(device)
    preds = []

    # THE LOOP WITH PROGRESS BAR
    for _ in tqdm(range(steps), desc="Torlin Progress", leave=False):
        with torch.no_grad():
            mu_e, _, mu_s, _ = enc(curr_w)
            z = torch.cat([mu_e, mu_s.reshape(1, -1)], dim=-1)
            y_h = pred(z)
            preds.append(y_h.item())
            
            new_r = curr_w[:, -1:, :].clone()
            new_r[:, :, 0] = y_h 
            curr_w = torch.cat([curr_w[:, 1:, :], new_r], dim=1)
    
    y_p, y_t = np.array(preds).ravel(), np.array(y_ground_truth).ravel()
    L = min(len(y_p), len(y_t))
    y_p, y_t = y_p[:L], y_t[:L]
    
    metrics = {
        "total_rmse": root_mean_squared_error(y_t, y_p),
        "start_rmse": root_mean_squared_error(y_t[:24], y_p[:24]) if L>=24 else 0,
        "end_rmse": root_mean_squared_error(y_t[-24:], y_p[-24:]) if L>=24 else 0,
        "cumulative_rmse": [root_mean_squared_error(y_t[:t], y_p[:t]) for t in range(1, L + 1)]
    }
    return y_p, y_t, metrics

def evaluate_sphlin_recursive(enc_e, enc_s, pred, XL_w, XC_w, y_gt, p, steps=250, device="cuda"):
    if enc_e: enc_e.eval()
    if enc_s: enc_s.eval()
    pred.eval()
    curr_L, curr_C = XL_w.clone().to(device), XC_w.clone().to(device)
    preds = []

    # THE LOOP WITH PROGRESS BAR
    for _ in tqdm(range(steps), desc="Sphlin Progress", leave=False):
        with torch.no_grad():
            z_p = []
            if enc_e: z_p.append(enc_e(curr_L)[0])
            if enc_s: z_p.append(enc_s(curr_C)[0])
            z = torch.cat(z_p, dim=-1)
            y_h = pred(z)
            preds.append(y_h.item())
            
            new_L = curr_L[:, -1:, :].clone()
            new_L[:, :, 0] = y_h 
            curr_L = torch.cat([curr_L[:, 1:, :], new_L], dim=1)
            new_C = curr_C[:, -1:, :].clone()
            curr_C = torch.cat([curr_C[:, 1:, :], new_C], dim=1)

    y_p, y_t = np.array(preds).ravel(), np.array(y_gt).ravel()
    L = min(len(y_p), len(y_t))
    y_p, y_t = y_p[:L], y_t[:L]

    metrics = {
        "total_rmse": root_mean_squared_error(y_t, y_p),
        "start_rmse": root_mean_squared_error(y_t[:24], y_p[:24]) if L>=24 else 0,
        "end_rmse": root_mean_squared_error(y_t[-24:], y_p[-24:]) if L>=24 else 0,
        "cumulative_rmse": [root_mean_squared_error(y_t[:t], y_p[:t]) for t in range(1, L + 1)]
    }
    return y_p, y_t, metrics

def plot_comparative_forecast(y_true, y_pred_tor, y_pred_sph, metrics_tor, metrics_sph, p, steps):
    """
    Plots the full recursive forecast comparison for a specific horizon.
    """
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), gridspec_kw={'height_ratios': [2, 1]})
    
    # --- Top Plot: The Forecast ---
    ax1.plot(y_true, label="Ground Truth", color="black", alpha=0.3, linestyle="--")
    ax1.plot(y_pred_tor, label=f"Torlin (Toroidal) - End RMSE: {metrics_tor['end_rmse']:.4f}", color="blue", linewidth=1.5)
    ax1.plot(y_pred_sph, label=f"Baseline (Euclidean) - End RMSE: {metrics_sph['end_rmse']:.4f}", color="green", linewidth=1.5)
    
    # Boundary line for training history
    ax1.axvline(x=p.window_size, color='red', linestyle=':', label=f"Training Horizon ({p.window_size}h)")
    
    ax1.set_title(f"Recursive Forecast Stability Battle ({steps} Steps)", fontsize=14)
    ax1.set_ylabel("Normalized Value")
    ax1.legend(loc='upper right')
    ax1.grid(alpha=0.2)
    
    # --- Bottom Plot: The Error Growth ---
    ax2.plot(metrics_tor['cumulative_rmse'], label="Torlin Error Growth", color="blue")
    ax2.plot(metrics_sph['cumulative_rmse'], label="Baseline (Euclidean) Error Growth", color="green")
    
    ax2.set_title("Quantifying the Drift (Cumulative RMSE)", fontsize=12)
    ax2.set_xlabel("Forecast Horizon (Steps)")
    ax2.set_ylabel("RMSE")
    ax2.legend()
    ax2.grid(alpha=0.2)
    
    plt.tight_layout()
    plt.show()


# --- FULL EVALUATION BATTLE: Torlin vs. Sphlin ---

# 1. Configuration
idx = 0 
HORIZON = 1000 
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Run & Unpack Torlin (Joint)
print(f"Training Torlin (Joint Toroidal)... Target Horizon: {HORIZON}")
res_t = Torlin.run_torlin(X_train, X_test, y_train, y_test, p, 
                          num_periods=num_distinct_periods, sliding_size=sliding_size)
_, _, _, _, (_, _, y_test_w_t), _, _, (enc_t, pred_t, X_te_w_t) = res_t

# 3. Run & Unpack Sphlin (Euclidean SOTA)
print("Training Sphlin (Euclidean SOTA)...")
res_s = Sphlin.run_sphlin_LSTM(X_train, X_test, y_train, y_test, p, 
                               sliding_size=sliding_size)
_, _, _, _, (_, _, y_test_w_s), _, _, (enc_e_s, enc_s_s, pred_s, XL_te_s, XC_te_s) = res_s

# 4. Generate Recursive Forecasts
print(f"Generating recursive forecasts...")

y_p_t, y_t_t, m_t = evaluate_recursive_forecast(
    enc_t, pred_t, X_te_w_t[idx:idx+1], y_test_w_t[idx:idx+HORIZON], p, steps=HORIZON, device=device)

y_p_s, y_t_s, m_s = evaluate_sphlin_recursive(
    enc_e_s, enc_s_s, pred_s, XL_te_s[idx:idx+1], XC_te_s[idx:idx+1], y_test_w_s[idx:idx+HORIZON], p, steps=HORIZON, device=device)

# 5. Comparative Print Statements
# Ensuring start_rmse exists for Sphlin if the function missed it
if 'start_rmse' not in m_s:
    m_s['start_rmse'] = root_mean_squared_error(y_t_s[:24], y_p_s[:24])

print("\n" + "="*45)
print(f"{'METRIC':<22} | {'TORLIN':<10} | {'SPHLIN':<10}")
print("-" * 45)
print(f"{'Global RMSE':<22} | {m_t['total_rmse']:<10.4f} | {m_s['total_rmse']:<10.4f}")
print(f"{'Short-term (First 24h)':<22} | {m_t['start_rmse']:<10.4f} | {m_s['start_rmse']:<10.4f}")
print(f"{'Long-term (Last 24h)':<22} | {m_t['end_rmse']:<10.4f} | {m_s['end_rmse']:<10.4f}")
print("="*45)

# 6. Final Visualization
plot_comparative_forecast(y_t_t, y_p_t, y_p_s, m_t, m_s, p, HORIZON)

In [ ]:
# -----------------------------
# SEMI-SUPERVISED (80% missing)
# -----------------------------
MISSING_LABEL_PCT = 0.80

print(f"--- Semi-Supervised Task: H=1 Prediction ({int(MISSING_LABEL_PCT*100)}% Missing) ---")

# --- SCALE ---
X_tr_s, X_te_s = scale_train_and_test_sets(X_train, X_test)
y_tr_s, y_te_s = scale_train_and_test_sets(y_train, y_test)

# --- WINDOWING ---
X_tr_w = Windowing.make_windows_from_X(X_tr_s, p.window_size, sliding_size).to(device)
X_te_w = Windowing.make_windows_from_X(X_te_s, p.window_size, sliding_size).to(device)
y_tr_w = Windowing.make_windows_from_y(y_tr_s, p.window_size, sliding_size, task=p.task).reshape(-1, 1)
y_te_w = Windowing.make_windows_from_y(y_te_s, p.window_size, sliding_size, task=p.task).reshape(-1, 1)

# --- MASK LABELS AFTER WINDOWING ---
keep_mask = torch.rand(len(y_tr_w)) > MISSING_LABEL_PCT
y_tr_w = y_tr_w.clone()
y_tr_w[~keep_mask] = torch.nan

# --- ALIGN LENGTHS ---
min_len = min(X_tr_w.size(0), y_tr_w.shape[0])
X_tr_w, y_tr_w = X_tr_w[:min_len], y_tr_w[:min_len]
X_te_min = min(X_te_w.size(0), y_te_w.shape[0])
X_te_w, y_te_w = X_te_w[:X_te_min], y_te_w[:X_te_min]

# --- DATA LOADERS ---
loader = DataLoader(TensorDataset(X_tr_w, y_tr_w), batch_size=p.batch_size, shuffle=True)

# --- TRAIN Sphlin (Euclidean baseline) ---
print("Training Sphlin (Euclidean SOTA)...")
res_s = Sphlin.run_sphlin_LSTM(X_train, X_test, y_tr_w.cpu().numpy(), y_test, p, sliding_size=sliding_size)
_, _, _, _, (_, _, _), _, _, (enc_e_s, enc_s_s, pred_s, XL_te_s, XC_te_s) = res_s

# --- TRAIN Torlin ---
print("Training Torlin...")
res_t = Torlin.run_torlin(X_train, X_test, y_tr_w.cpu().numpy(), y_test, p,
                          num_periods=num_distinct_periods, sliding_size=sliding_size)
_, _, _, _, (_, _, y_t_w), _, _, (enc_t, pred_t, X_te_w_t) = res_t

# --- PREDICTIONS ---
def get_h1_predictions(enc_type, models, data):
    with torch.no_grad():
        if enc_type == 'torlin':
            enc, pred, X = models[0], models[1], data.to(device)
            mu_e, _, mu_s, _ = enc(X)
            z = torch.cat([mu_e, mu_s.reshape(mu_s.shape[0], -1)], dim=-1)
            return pred(z).cpu().numpy().ravel()
        else:
            enc_e, enc_s, pred, XL, XC = models[0], models[1], models[2], data[0].to(device), data[1].to(device)
            z_parts = []
            if enc_e: z_parts.append(enc_e(XL)[0])
            if enc_s: z_parts.append(enc_s(XC)[0])
            return pred(torch.cat(z_parts, dim=-1)).cpu().numpy().ravel()

y_hat_t = get_h1_predictions('torlin', [enc_t, pred_t], X_te_w_t)
y_hat_s = get_h1_predictions('sphlin', [enc_e_s, enc_s_s, pred_s], [XL_te_s, XC_te_s])

# --- METRICS ---
rmse_t = root_mean_squared_error(y_t_w, y_hat_t)
rmse_s = root_mean_squared_error(y_t_w, y_hat_s)

print("\n" + "="*45)
print(f"{'H=1 RMSE':<22} | Torlin: {rmse_t:.4f} | Sphlin: {rmse_s:.4f}")
print("="*45)

plt.figure(figsize=(15, 5))
plt.plot(y_t_w[:500], label="Actual", alpha=0.3)
plt.plot(y_hat_t[:500], label=f"Torlin RMSE: {rmse_t:.4f}")
plt.plot(y_hat_s[:500], label=f"Sphlin RMSE: {rmse_s:.4f}")
plt.title(f"H=1 Semi-Supervised Performance ({int(MISSING_LABEL_PCT*100)}% Missing Labels)")
plt.legend()
plt.show()


In [ ]:
"💰 DIRECT PREDICTION"

rmse_direct, r2_direct, mae_direct = Predictors.make_direct_prediction(X_train, X_test, y_train, y_test,
                                                            sliding_size=sliding_size, prediction_task=prediction_task, p=p)
print(f"🎯😸 Direct pred Catboost. RMSE={rmse_direct:.3f}  MAE={mae_direct:.3f} R2={r2_direct:.3f}")

rmse_direct_mlp, r2_direct_mlp, mae_direct_mlp = Predictors.make_direct_prediction_mlp(X_train, X_test, y_train, y_test,
                                                            sliding_size=sliding_size, p=p)
print(f"🎯 Direct pred MLP RMSE={rmse_direct_mlp:.3f}  MAE={mae_direct_mlp:.3f} R2={r2_direct_mlp:.3f}")

metrics = {"rmse": rmse_direct_mlp, "r2": r2_direct_mlp, "mae": mae_direct_mlp,}
metrics = {k: round(float(v), 4) for k, v in metrics.items()}
append_run_to_csv(params=p.__dict__, metrics=metrics, dataset=dataset, file_path="geo_results.csv", method=f"direct",)


# print(f"\\val{{{rmse_direct:.3f}}}{{0}} & \\val{{{mae_direct:.3f}}}{{0}} & \\val{{{r2_direct:.3f}}}{{0}}")

# rmse_pca, r2_pca = make_direct_prediction_cov_pca(X_train, X_test, y_train, y_test, window_size=window_size,
#                                                   sliding_size=sliding_size, prediction_task=prediction_task,
#                                                   method="PCA")
# print(f"Covariance PCA -> RMSE: {rmse_pca:.4f}, R2: {r2_pca:.4f}")

# rmse_pga, r2_pga = make_direct_prediction_cov_pca(X_train, X_test, y_train, y_test, window_size=window_size,
#                                                   sliding_size=sliding_size, prediction_task=prediction_task,
#                                                   method="PGA")
# print(f"Covariance PGA -> RMSE: {rmse_pga:.4f}, R2: {r2_pga:.4f}")

# rmse_pca, r2_pca = make_direct_prediction_pca(X_train, X_test, y_train, y_test, window_size=window_size,
#                                               sliding_size=sliding_size, prediction_task=prediction_task)
# print(f"Test RMSE PCA: {rmse_pca:.4f}, R2 PCA: {r2_pca:.4f}")


In [ ]:
def plot_torus_latents(Z_test, period_idx=0):
    # Z_test is [Samples, Z_total]
    # Torus latents start after the Euclidean dims
    # Each period is a pair of (x, y)
    
    # Calculate start index of torus:
    z_sph_start = Z_test.shape[1] - (num_distinct_periods * 2)
    x_idx = z_sph_start + (period_idx * 2)
    y_idx = x_idx + 1
    
    plt.figure(figsize=(6,6))
    plt.scatter(Z_test[:, x_idx], Z_test[:, y_idx], alpha=0.5, s=2)
    plt.title(f"Latent Space: Torus Circle {period_idx}")
    plt.xlabel("Coordinate X (cos)")
    plt.ylabel("Coordinate Y (sin)")
    plt.axis('equal')
    plt.show()

def plot_3d_torus(Z_test, num_periods=2):
    # Z_test shape: [N, Z_total]
    # Torus dims are the LAST (num_periods * 2) dimensions
    z_sph_start = Z_test.shape[1] - (num_periods * 2)
    
    # Period 1 (Circle 1)
    theta = np.arctan2(Z_test[:, z_sph_start+1], Z_test[:, z_sph_start])
    
    # Period 2 (Circle 2) - Only if we have at least 2 periods
    if num_periods >= 2:
        phi = np.arctan2(Z_test[:, z_sph_start+3], Z_test[:, z_sph_start+2])
    else:
        # If only 1 period, we use a constant 0 for phi to show a ring in 3D
        phi = np.zeros_like(theta)
    
    # Torus Parametric Equations
    R, r = 2, 0.8  # R = distance from center to tube center, r = tube radius
    X = (R + r * np.cos(phi)) * np.cos(theta)
    Y = (R + r * np.cos(phi)) * np.sin(theta)
    Z = r * np.sin(phi)
    
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    # Use theta for color to show the wrap-around
    sc = ax.scatter(X, Y, Z, c=theta, cmap='hsv', s=2, alpha=0.6)
    
    # Make it look like a 3D box
    ax.set_zlim(-2, 2)
    ax.set_title(f"Learned Toroidal Manifold $T^{min(num_periods, 2)}$")
    plt.show()


# Run this after run_torlin
plot_torus_latents(Z_test, period_idx=0)
plot_3d_torus(Z_test)


In [ ]:
"🧠🕸️ part 2"

def plot_latent_hist(Z_train, Z_test=None, bins=80):
    Z_train = np.asarray(Z_train).flatten()
    plt.figure(figsize=(10,4))
    plt.hist(Z_train, bins=bins, alpha=0.6, label="Z_train")
    if Z_test is not None:
        Z_test = np.asarray(Z_test).flatten()
        plt.hist(Z_test, bins=bins, alpha=0.6, label="Z_test")
    plt.title("Latent histogram (train vs test)")
    plt.xlabel("latent value")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_kappa(logs):
    if "kappa" not in logs: 
        return
    plt.figure(figsize=(6,3))
    plt.plot(logs["kappa"])
    plt.title("κ over epochs")
    plt.xlabel("epoch")
    plt.ylabel("κ")
    plt.tight_layout()
    plt.show()

def plot_loss_curves(logs):
    keys = [k for k in logs if k in ["total", "recon", "kl_e", "kl_s", "pred"]]
    plt.figure(figsize=(10,4))
    for k in keys:
        plt.plot(logs[k], label=k)
    plt.title("Loss curves")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_latent_scatter(Z, dims=(0,1), n=5000):
    Z = np.asarray(Z)[:n]
    plt.figure(figsize=(5,4))
    plt.scatter(Z[:, dims[0]], Z[:, dims[1]], s=3, alpha=0.5)
    plt.title(f"Latent scatter: dim {dims[0]} vs {dims[1]}")
    plt.xlabel(f"dim {dims[0]}")
    plt.ylabel(f"dim {dims[1]}")
    plt.tight_layout()
    plt.show()

def plot_recon_vs_true(y_true, y_hat, n=500):
    y_true = np.asarray(y_true).reshape(-1)[:n]
    y_hat = np.asarray(y_hat).reshape(-1)[:n]
    plt.figure(figsize=(5,4))
    plt.scatter(y_true, y_hat, s=3, alpha=0.5)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--')
    plt.title("y_test vs y_pred")
    plt.xlabel("true")
    plt.ylabel("pred")
    plt.tight_layout()
    plt.show()

def plot_feature_cyclicity(X, top_k=3):
    from numpy.fft import rfft
    if hasattr(X, "to_numpy"):
        X = X.to_numpy()
    D = X.shape[1]
    cols = min(4, D)
    rows = (D + cols - 1) // cols

    plt.figure(figsize=(4*cols, 3*rows))
    for d in range(D):
        x = X[:, d] - X[:, d].mean()
        fft_vals = rfft(x)
        power = np.abs(fft_vals)**2
        power[0]=0
        idx = np.argsort(power)[-top_k:][::-1]
        ratios = power[idx] / (power.sum()+1e-8)

        ax = plt.subplot(rows, cols, d+1)
        ax.bar(range(top_k), ratios)
        ax.set_title(f"feat {d}")
        ax.set_xlabel("top k cycles")
        ax.set_ylabel("power ratio")

    plt.tight_layout()
    plt.show()

def show_cyclicity_per_feature(X, top_k=3, threshold=0.1):
    if hasattr(X, "to_numpy"):
        X = X.to_numpy()
    D = X.shape[1]
    for d in range(D):
        x = X[:, d]
        periods = compute_multicyclicity_scores(x, top_k=top_k, threshold=threshold)
        print(f"feature {d}: {periods}")




show_cyclicity_per_feature(X_train, top_k=3, threshold=0.1)

# plot
X_train_np = X_train.to_numpy()
plot_latent_hist(Z_train, Z_test)
plot_latent_scatter(Z_train, dims=(0,1))
plot_kappa(logs)
plot_loss_curves(logs)
plot_feature_cyclicity(X_train_np, top_k=3)
plot_recon_vs_true(y_test_w, y_hat, n=500)



In [ ]:
"🧠 sanity check"
from statsmodels.tsa.stattools import acf

def r2_z_to_y(Z_train, y_tr_w, Z_test, y_te_w):
    assert Z_train.shape[0] == y_tr_w.shape[0]

    y_hat = fit_catboost_multi(Z_train, y_tr_w, Z_test)
    r2    = r2_score(y_te_w, y_hat)
    return r2

def horizon_r2(Z, y_w, max_k=24):
    out = {}
    n   = len(Z)
    # Ensure we have enough data to actually fit a model
    for k in range(1, min(max_k + 1, n // 2)): 
        yk = y_w[k:]
        zk = Z[:-k]
        # Validation set must be large enough for CatBoost
        z_val = Z[-k:]
        y_val = y_w[-k:]
        if z_val.shape[0] < 2: # CatBoost requirement
            continue
        out[k] = r2_z_to_y(zk, yk, z_val, y_val)
    return out

def acf_distance(y_true_w, y_hat_w, nlags=40):
    y_true = y_true_w.mean(axis=1)
    y_hat  = y_hat_w.mean(axis=1)
    return np.linalg.norm(acf(y_true, nlags=nlags) - acf(y_hat, nlags=nlags))

def phase_alignment(z: Union[np.ndarray, torch.Tensor], t: np.ndarray, omega: float) -> float:
    """Compute phase alignment between z and sin/cos at frequency omega."""
    if isinstance(z, torch.Tensor):
        z = z.detach().cpu().numpy()
    sin_theta = np.sin(omega * t)
    cos_theta = np.cos(omega * t)
    return max(abs(np.corrcoef(z, sin_theta)[0,1]), abs(np.corrcoef(z, cos_theta)[0,1]))


def plot_sanity_all(r2, horizon_r2_dict, y_test_w, y_hat_w, scores, z_euc: int, period=None):
    """Single figure with 4 panels:
    1) scalar R²
    2) horizon R² curve
    3) ACF true vs pred (mean over windows)
    4) phase alignment per latent dim

    z_euc = # euclidean latent dims. note that the no-euclidean dim (spherical or toroidal) isnt needed, we can infer by 
    z_noneuc = len(scores) - z_euc"""
    import matplotlib.patches as mpatches

    y_true_1d = np.asarray(y_test_w).mean(axis=1)
    y_hat_1d  = np.asarray(y_hat_w).mean(axis=1)

    # ACF (manual, 1D)
    def acf1d(x, nlags=40):
        x    = x - x.mean()
        corr = np.correlate(x, x, mode="full")
        corr = corr[corr.size // 2:]
        corr = corr / corr[0]
        return corr[:nlags + 1]

    acf_t = acf1d(y_true_1d, nlags=200)
    acf_h = acf1d(y_hat_1d, nlags=200)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    ax1, ax2, ax3, ax4 = axes.ravel()

    # 1. Scalar R² 'plot'
    ax1.text(0.5, 0.5, f"R² (Z→Y) = {r2:.4f}", ha="center", va="center", fontsize=14)
    ax1.axis("off")

    # 2. Horizon R² plot
    ax2.plot(list(horizon_r2_dict.keys()), list(horizon_r2_dict.values()))
    ax2.set_xlabel("Horizon k")
    ax2.set_ylabel("R²")
    ax2.set_title("Horizon R²")

    # 3. ACF (= autocorrelation function) plot
    ax3.plot(acf_t, label="true")
    ax3.plot(acf_h, label="pred")
    ax3.set_xlabel("Lag")
    ax3.set_ylabel("ACF")
    ax3.set_title("ACF (mean over window)")
    ax3.legend()

    # 4. Phase alignment plot
    # colors = ["C0"] * (len(scores) - z_sph) + ["orange"] * z_sph
    colors = ["blue"] * z_euc + ["orange"] * (len(scores) - z_euc)
    ax4.bar(range(len(scores)), scores, color=colors)
    ax4.set_xlabel("latent dim")
    ax4.set_ylabel("phase alignment")
    ax4.set_title(f"Phase alignment (period={period})" if period is not None else "Phase alignment")

    blue_patch   = mpatches.Patch(color="blue", label=r"$z_{euc}$")
    orange_patch = mpatches.Patch(color="orange", label=r"$z_{non-euc}$")
    ax4.legend(handles=[blue_patch, orange_patch])

    plt.tight_layout()
    plt.show()

r2       = r2_z_to_y(Z_train, y_train_w, Z_test, y_test_w)
out      = horizon_r2(Z_test, y_test_w, max_k=24)
acf_dist = acf_distance(y_test_w, y_hat, nlags=150)

t      = np.arange(len(Z_train))
x      = y_train_w.mean(axis=1)
period = Periodicity._dominant_periods(x, fs=1.0, topk=1, max_period=400)[0]
omega  = 2 * np.pi / period
scores = [phase_alignment(Z_train[:, i], t, omega) for i in range(Z_train.shape[1])]

plot_sanity_all(r2, out, y_test_w, y_hat, scores, z_euc=z_dim_euclid, period=period)


In [ ]:
"📈 Visualize dataset"

def plot_1_feature_on_separate_plot(X_feature, col_name: str | None = None, max_samples: int = 50_000, stride: int = 5, color: str = 'blue'):
    plt.figure(figsize=(14, 4))
    # col_smooth = X_feature.rolling(window=2).mean()

    y_data = X_feature[:max_samples:stride] if isinstance(X_feature, np.ndarray) else X_feature.iloc[:max_samples:stride].values
    plt.plot(y_data, linewidth=0.7, alpha=0.7, color=color)

    plt.margins(x=0.01)
    # plt.xlim(0, len(y_data) * 1.05)

    plt.title(col_name if col_name is not None else "y-axis")
    plt.xlabel("Time idx [#]")
    plt.grid()
    plt.tight_layout()
    plt.show()

stride = 10
max_samples = 1_000_000

plot_1_feature_on_separate_plot(y, max_samples=max_samples, stride=stride, color='orange')

for col in X.columns:
    if X[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    plot_1_feature_on_separate_plot(X[col], col_name=col, max_samples=max_samples, stride=stride)


In [ ]:
# ====== code breaker  ========
1 = rf


In [ ]:
"[🍩 TorLin + OldTor] torus methods"

# class Torlin2:
#     @staticmethod
#     def sample_vmf_approximate(mu, kappa):
#         "Approximate vMF sampler (S^1 sampling in 2D)"
#         noise = torch.randn_like(mu) * (1.0/kappa)
#         return F.normalize(mu + noise, dim=-1)

#     @staticmethod
#     def sample_vmf_exact(mu_xy: torch.Tensor, kappa: torch.Tensor):
#         """Exact vMF sampler on S¹ using VonMises.
#         mu_xy: (B, 2)
#         kappa: (B,)"""
#         mu_xy    = F.normalize(mu_xy, dim=-1)
#         mu_angle = torch.atan2(mu_xy[:, 1], mu_xy[:, 0])
#         # theta    = torch.distributions.VonMises(mu_angle, kappa).rsample()
#         theta    = torch.distributions.VonMises(mu_angle, kappa).sample()
#         return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)

#     @staticmethod
#     def vmf_kl_s1(kappa: torch.Tensor):
#         """Exact KL(vMF || Uniform) on S¹."""
#         i0 = torch.special.i0(kappa)
#         i1 = torch.special.i1(kappa)
#         return kappa * (i1 / (i0 + 1e-8)) - torch.log(i0 + 1e-8)

#     @staticmethod # this is the good one (works wwith the old versio nof lstm encoder)! but gives warnings
#     def X_run_torlin_pipeline(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
#                               y_test: np.ndarray, p: RunParams, angular_features_list: list) -> Tuple:
#         """Full Toroidal VAE pipeline with Reconstruction Decoder and original latent dimension splitting logic."""
#         if len(angular_features_list) > 0:
#             X_cyc_train = X_train[angular_features_list]
#             X_cyc_test = X_test[angular_features_list]
#             X_linear_train = X_train.drop(columns=angular_features_list)
#             X_linear_test = X_test.drop(columns=angular_features_list)
#         else:
#             X_cyc_train = pd.DataFrame()
#             X_cyc_test = pd.DataFrame()
#             X_linear_train = X_train.copy()
#             X_linear_test = X_test.copy()

#         num_cyc_features = X_cyc_train.shape[1]
#         z_dim_torus = 2 * num_cyc_features
#         z_dim_euclid = p.z_dim_total - z_dim_torus
#         if z_dim_euclid < 8:
#             z_dim_euclid = 8
#         print(f"Latent Split: Euclid={z_dim_euclid}, Torus Circles={num_cyc_features}, Total Dim={z_dim_euclid + z_dim_torus}")

#         X_lin_train, X_lin_test = scale_train_and_test_sets(X_linear_train, X_linear_test)
#         if num_cyc_features > 0:
#             X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)
#         else:
#             X_cyc_train = torch.empty((len(X_lin_train), 0), dtype=torch.float32)
#             X_cyc_test = torch.empty((len(X_lin_test), 0), dtype=torch.float32)

#         X_lin_tr_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size).to(device)
#         X_cyc_tr_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size).to(device)
#         X_lin_te_w = Windowing.make_windows_from_X(X_lin_test, p.window_size, p.sliding_size).to(device)
#         X_cyc_te_w = Windowing.make_windows_from_X(X_cyc_test, p.window_size, p.sliding_size).to(device)

#         hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
#         enc_e = LSTMEncoderEuclid(X_lin_train.shape[1], hidden_dim_split, z_dim_euclid).to(device)
#         enc_t = LSTMToroidalEncoder(X_cyc_train.shape[1], hidden_dim_split, num_cyc_features).to(device) if num_cyc_features > 0 else None

#         output_dim = X_lin_train.shape[1] + X_cyc_train.shape[1]
#         decoder = MLPDecoder(z_dim_total=z_dim_euclid + z_dim_torus, window_size=p.window_size, output_dim=output_dim, hidden_dim=p.hidden_dim).to(device)
#         optimizer = torch.optim.AdamW(list(enc_e.parameters()) + (list(enc_t.parameters()) if enc_t else []) + list(decoder.parameters()), lr=p.lr_optimizer)

#         loader = DataLoader(TensorDataset(X_lin_tr_w, X_cyc_tr_w), batch_size=p.batch_size, shuffle=True)
#         lambdas = {'recon': p.lambda_recon, 'euc': p.lambda_latent / np.sqrt(z_dim_euclid), 'tor': p.lambda_latent / np.sqrt(z_dim_torus if z_dim_torus > 0 else 1)}
#         criterion = nn.MSELoss()
#         best_loss = float("inf")
#         counter = 0

#         for epoch in range(p.epochs):
#             enc_e.train()
#             if enc_t: enc_t.train()
#             decoder.train()
#             epoch_loss = 0

#             for b_lin, b_cyc in loader:
#                 optimizer.zero_grad()
#                 b_lin, b_cyc = b_lin.to(device), b_cyc.to(device)

#                 mu_e, logvar_e = enc_e(b_lin)
#                 if mu_e.dim() == 3:
#                     mu_e = mu_e[:, -1, :]
#                     logvar_e = logvar_e[:, -1, :]
#                 z_e = Reparam.reparam_gaussian(mu_e, logvar_e)

#                 if enc_t:
#                     mu_t, kappa_t = enc_t(b_cyc)
#                     if mu_t.dim() == 4:
#                         mu_t = mu_t[:, -1, :, :]
#                         kappa_t = kappa_t[:, -1, :]
#                     elif mu_t.dim() == 3:
#                         mu_t = mu_t[:, -1, :].unsqueeze(1)
#                         kappa_t = kappa_t[:, -1].unsqueeze(1)
#                     elif mu_t.dim() == 2:
#                         mu_t = mu_t.unsqueeze(1)
#                         kappa_t = kappa_t.unsqueeze(1)

#                     n_cyc_enc = mu_t.shape[1]
#                     if n_cyc_enc != num_cyc_features:
#                         print("WARNING: mismatch", num_cyc_features, "vs", n_cyc_enc)

#                     z_t_list = [Torlin.sample_vmf_exact(mu_t[:, i, :], kappa_t[:, i]) for i in range(min(n_cyc_enc, num_cyc_features))]
#                     if len(z_t_list) < num_cyc_features:
#                         pad = torch.zeros((b_lin.size(0), 2 * (num_cyc_features - len(z_t_list))), device=device)
#                         z_t = torch.cat(z_t_list + [pad], dim=-1)
#                     else:
#                         z_t = torch.cat(z_t_list[:num_cyc_features], dim=-1)
#                 else:
#                     z_t = torch.zeros((b_lin.size(0), z_dim_torus), device=device)

#                 z_combined = torch.cat([z_e, z_t], dim=-1)
#                 recon = decoder(z_combined)

#                 target = torch.cat([b_lin, b_cyc], dim=-1)
#                 loss_recon = criterion(recon.view(recon.size(0), -1), target.view(target.size(0), -1))
#                 kl_euc = -0.5 * torch.sum(1 + logvar_e - mu_e.pow(2) - logvar_e.exp(), dim=1).mean()
#                 kl_tor = Torlin.vmf_kl_s1(kappa_t).mean() if enc_t else 0

#                 total_loss = lambdas['recon'] * loss_recon + lambdas['euc'] * kl_euc + lambdas['tor'] * kl_tor
#                 total_loss.backward()
#                 optimizer.step()
#                 epoch_loss += total_loss.item()

#             avg_epoch_loss = epoch_loss / len(loader)
#             best_loss, counter, stop = early_stop(avg_epoch_loss, best_loss, counter, p.earlystop_patience)
#             if stop:
#                 break

#         enc_e.eval()
#         if enc_t: enc_t.eval()

#         with torch.no_grad():
#             mu_e_te, _ = enc_e(X_lin_te_w)
#             if mu_e_te.dim() == 3:
#                 mu_e_te = mu_e_te[:, -1, :]
#             if enc_t:
#                 mu_t_te, _ = enc_t(X_cyc_te_w)
#                 if mu_t_te.dim() == 4:
#                     mu_t_te = mu_t_te[:, -1, :, :]
#                 elif mu_t_te.dim() == 3:
#                     mu_t_te = mu_t_te[:, -1, :].unsqueeze(1)
#                 elif mu_t_te.dim() == 2:
#                     mu_t_te = mu_t_te.unsqueeze(1)

#                 z_t_te = torch.cat([F.normalize(mu_t_te[:, i, :], dim=-1) for i in range(min(mu_t_te.shape[1], num_cyc_features))], dim=-1)
#                 if z_t_te.shape[1] < z_dim_torus:
#                     pad = torch.zeros((z_t_te.size(0), z_dim_torus - z_t_te.shape[1]), device=device)
#                     z_t_te = torch.cat([z_t_te, pad], dim=-1)
#             else:
#                 z_t_te = torch.zeros((mu_e_te.size(0), z_dim_torus), device=device)

#             Z_test = torch.cat([mu_e_te, z_t_te], dim=-1).cpu().numpy()

#             mu_e_tr, _ = enc_e(X_lin_tr_w)
#             if mu_e_tr.dim() == 3:
#                 mu_e_tr = mu_e_tr[:, -1, :]
#             if enc_t:
#                 mu_t_tr, _ = enc_t(X_cyc_tr_w)
#                 if mu_t_tr.dim() == 4:
#                     mu_t_tr = mu_t_tr[:, -1, :, :]
#                 elif mu_t_tr.dim() == 3:
#                     mu_t_tr = mu_t_tr[:, -1, :].unsqueeze(1)
#                 elif mu_t_tr.dim() == 2:
#                     mu_t_tr = mu_t_tr.unsqueeze(1)

#                 z_t_tr = torch.cat([F.normalize(mu_t_tr[:, i, :], dim=-1) for i in range(min(mu_t_tr.shape[1], num_cyc_features))], dim=-1)
#                 if z_t_tr.shape[1] < z_dim_torus:
#                     pad = torch.zeros((z_t_tr.size(0), z_dim_torus - z_t_tr.shape[1]), device=device)
#                     z_t_tr = torch.cat([z_t_tr, pad], dim=-1)
#             else:
#                 z_t_tr = torch.zeros((mu_e_tr.size(0), z_dim_torus), device=device)

#             Z_train = torch.cat([mu_e_tr, z_t_tr], dim=-1).cpu().numpy()

#         y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
#         y_train_win = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
#         y_test_win = Windowing.make_windows_from_y(y_test_s, p.window_size, p.sliding_size, task=p.task)

#         y_hat = fit_catboost_multi(Z_train, y_train_win, Z_test)
#         rmse  = root_mean_squared_error(y_test_win, y_hat)
#         r2    = r2_score(y_test_win, y_hat)
#         mae   = mean_absolute_error(y_test_win, y_hat)
#         return rmse, r2, mae, (Z_train, Z_test)

#     @staticmethod  # old (no warmup)
#     def X_train_torlin(enc_e, enc_t, decoder, loader, optimizer, p, lambdas, num_cyc_features, z_dim_torus, device):
#         criterion = nn.MSELoss()
#         best_loss, counter = float("inf"), 0
        
#         for epoch in range(p.epochs):
#             enc_e.train(); decoder.train()
#             if enc_t: enc_t.train()
#             epoch_loss = 0
#             total_kl_tor = 0

#             for b_lin, b_cyc in loader:
#                 optimizer.zero_grad()
#                 b_lin, b_cyc = b_lin.to(device), b_cyc.to(device)

#                 # 1. Encode
#                 mu_e, logvar_e = enc_e(b_lin)
#                 z_e = Reparam.reparam_gaussian(mu_e, logvar_e)

#                 if enc_t:
#                     mu_t, kappa_t = enc_t(b_cyc)
#                     # Sample each circle on the torus
#                     z_t = torch.cat([Torlin.sample_vmf_exact(mu_t[:, i, :], kappa_t[:, i]) 
#                                     for i in range(num_cyc_features)], dim=-1)
#                     kl_tor = Torlin.vmf_kl_s1(kappa_t).mean()
#                     total_kl_tor += kl_tor.item()
#                 else:
#                     z_t = torch.zeros((b_lin.size(0), z_dim_torus), device=device)
#                     kl_tor = 0

#                 # 2. Decode & Loss
#                 z_combined = torch.cat([z_e, z_t], dim=-1)
#                 recon = decoder(z_combined)
                
#                 target = torch.cat([b_lin, b_cyc], dim=-1).view(b_lin.size(0), -1)
#                 loss_recon = criterion(recon.view(recon.size(0), -1), target)
#                 kl_euc = -0.5 * torch.sum(1 + logvar_e - mu_e.pow(2) - logvar_e.exp(), dim=1).mean()
                
#                 total_loss = (lambdas['recon'] * loss_recon + 
#                             lambdas['euc'] * kl_euc + 
#                             lambdas['tor'] * kl_tor)
                
#                 total_loss.backward()
#                 optimizer.step()
#                 epoch_loss += total_loss.item()

#             avg_loss = epoch_loss / len(loader)
            
#             # Monitor progress every 10 epochs
#             if epoch % 10 == 0 and enc_t:
#                 print(f"Epoch {epoch} | KL Tor: {total_kl_tor/len(loader):.4f} | Recon: {avg_loss:.4f}")

#             best_loss, counter, stop = early_stop(avg_loss, best_loss, counter, p.earlystop_patience)
#             if stop: 
#                 break
#         return enc_e, enc_t, decoder

#     @staticmethod
#     def _train_torlin(enc_e, enc_t, decoder, loader, optimizer, p, lambdas, num_cyc_features, z_dim_torus, device, use_warmup=True):
#         criterion = nn.MSELoss(reduction='none')
#         best_loss, counter = float("inf"), 0
#         total_steps = p.epochs * len(loader)
#         current_step = 0
        
#         for epoch in range(p.epochs):
#             enc_e.train(); decoder.train()
#             if enc_t: enc_t.train()
            
#             epoch_mse_lin, epoch_mse_cyc = 0, 0
#             epoch_kl_euc, epoch_kl_tor = 0, 0

#             for b_lin, b_cyc in loader:
#                 optimizer.zero_grad()
#                 b_lin, b_cyc = b_lin.to(device), b_cyc.to(device)
#                 current_step += 1
#                 beta = min(1.0, current_step / (0.3 * total_steps + 1e-9)) if use_warmup else 1.0

#                 # 1. Encode
#                 mu_e, logvar_e = enc_e(b_lin)
#                 z_e = Reparam.reparam_gaussian(mu_e, logvar_e)
                
#                 mu_t, kappa_t = enc_t(b_cyc)
#                 z_t = torch.cat([Torlin.sample_vmf_exact(mu_t[:, i, :], kappa_t[:, i]) for i in range(num_cyc_features)], dim=-1)
                
#                 # 2. Decode
#                 z_combined = torch.cat([z_e, z_t], dim=-1)
#                 recon = decoder(z_combined)
                
#                 # 3. Component Losses
#                 target = torch.cat([b_lin, b_cyc], dim=-1).view(b_lin.size(0), -1)
#                 raw_mse = criterion(recon.view(recon.size(0), -1), target)
                
#                 flat_dim_lin = b_lin.shape[1] * b_lin.shape[2]
#                 mse_lin = raw_mse[:, :flat_dim_lin].mean()
#                 mse_cyc = raw_mse[:, flat_dim_lin:].mean()
                
#                 kl_euc = -0.5 * torch.sum(1 + logvar_e - mu_e.pow(2) - logvar_e.exp(), dim=1).mean()
#                 kl_tor = Torlin.vmf_kl_s1(kappa_t).mean()

#                 loss = (lambdas['recon'] * (mse_lin + mse_cyc)) + \
#                     (lambdas['euc'] * beta * kl_euc) + \
#                     (lambdas['tor'] * beta * kl_tor)
                
#                 loss.backward()
#                 optimizer.step()
                
#                 epoch_mse_lin += mse_lin.item()
#                 epoch_mse_cyc += mse_cyc.item()
#                 epoch_kl_euc += kl_euc.item()
#                 epoch_kl_tor += kl_tor.item()

#             if epoch % 10 == 0:
#                 L = len(loader)
#                 print(f"Epoch {epoch} | MSE(lin/cyc): {epoch_mse_lin/L:.3f}/{epoch_mse_cyc/L:.3f} | KL(euc/tor): {epoch_kl_euc/L:.3f}/{epoch_kl_tor/L:.4f}")

#         return enc_e, enc_t, decoder

#     @staticmethod
#     def run_torlin_pipeline(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray, 
#                             y_test: np.ndarray, p: RunParams, angular_features_list: list, epsilon=1e-3) -> Tuple:
#         # 1. Feature Splitting
#         if len(angular_features_list) > 0:
#             X_cyc_train    = X_train[angular_features_list]
#             X_cyc_test     = X_test[angular_features_list]
#             X_linear_train = X_train.drop(columns=angular_features_list)
#             X_linear_test  = X_test.drop(columns=angular_features_list)
#         else:
#             X_cyc_train, X_cyc_test       = pd.DataFrame(), pd.DataFrame()
#             X_linear_train, X_linear_test = X_train.copy(), X_test.copy()

#         num_cyc_features = X_cyc_train.shape[1]
#         z_dim_torus      = 2 * num_cyc_features
#         z_dim_euclid     = max(8, p.z_dim_total - z_dim_torus)

#         # 2. Scaling & Windowing
#         X_lin_train, X_lin_test = scale_train_and_test_sets(X_linear_train, X_linear_test)
#         if num_cyc_features > 0:
#             X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)
#         else:
#             X_cyc_train = torch.empty((len(X_lin_train), 0))
#             X_cyc_test  = torch.empty((len(X_lin_test), 0))

#         X_lin_tr_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size).to(device)
#         X_cyc_tr_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size).to(device)
#         X_lin_te_w = Windowing.make_windows_from_X(X_lin_test, p.window_size, p.sliding_size).to(device)
#         X_cyc_te_w = Windowing.make_windows_from_X(X_cyc_test, p.window_size, p.sliding_size).to(device)

#         # 3. Model Init
#         hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
#         enc_e = LSTMEncoderEuclid(X_lin_train.shape[1], hidden_dim_split, z_dim_euclid).to(device)
#         enc_t = LSTMToroidalEncoder(X_cyc_train.shape[1], hidden_dim_split, num_cyc_features, n_layers=3, epsilon=epsilon).to(device) if num_cyc_features > 0 else None

#         output_dim = X_lin_train.shape[1] + X_cyc_train.shape[1]
#         decoder    = MLPDecoder(z_dim_total=z_dim_euclid + z_dim_torus, window_size=p.window_size, 
#                                 output_dim=output_dim, hidden_dim=p.hidden_dim).to(device)
        
#         params    = list(enc_e.parameters()) + list(decoder.parameters())
#         if enc_t: params += list(enc_t.parameters())
#         optimizer = torch.optim.AdamW(params, lr=p.lr_optimizer)

#         # 4. Training Loop
#         loader  = DataLoader(TensorDataset(X_lin_tr_w, X_cyc_tr_w), batch_size=p.batch_size, shuffle=True)
#         lambdas = {'recon': p.lambda_recon,
#                 'euc': p.lambda_latent / np.sqrt(z_dim_euclid),
#                 'tor': p.lambda_latent / np.sqrt(z_dim_torus if z_dim_torus > 0 else 1)}
#         enc_e, enc_t, decoder = Torlin._train_torlin(enc_e, enc_t, decoder, loader, optimizer, p, lambdas, num_cyc_features, z_dim_torus, device)

#         # 5. Inference
#         enc_e.eval()
#         if enc_t: enc_t.eval()
        
#         with torch.no_grad():
#             def get_z(l_w, c_w):
#                 mu_e_val, _ = enc_e(l_w)
#                 if enc_t:
#                     mu_t_val, _ = enc_t(c_w)
#                     # For inference, we use the normalized mean direction (mu)
#                     z_t_val = torch.cat([F.normalize(mu_t_val[:, i, :], dim=-1) 
#                                         for i in range(num_cyc_features)], dim=-1)
#                 else:
#                     z_t_val = torch.zeros((mu_e_val.size(0), z_dim_torus), device=device)
#                 return torch.cat([mu_e_val, z_t_val], dim=-1).cpu().numpy()

#             Z_train = get_z(X_lin_tr_w, X_cyc_tr_w)
#             Z_test  = get_z(X_lin_te_w, X_cyc_te_w)

#         # 6. Downstream
#         y_tr_s, y_te_s = scale_train_and_test_sets(y_train, y_test)
#         y_tr_w = Windowing.make_windows_from_y(y_tr_s, p.window_size, p.sliding_size, task=p.task)
#         y_te_w = Windowing.make_windows_from_y(y_te_s, p.window_size, p.sliding_size, task=p.task)

#         y_hat = fit_catboost_multi(Z_train, y_tr_w, Z_test)
#         rmse  = root_mean_squared_error(y_te_w, y_hat)
#         r2    = r2_score(y_te_w, y_hat)
#         mae   = mean_absolute_error(y_te_w, y_hat)

#         PlotTopolin2.plot_kappa_dist(enc_t, loader, device)
#         return rmse, r2, mae, (Z_train, Z_test)

# class PlotTopolin2:
#     @staticmethod
#     def old_plot_torus(Z_numpy, z_dim_euc, feat_a=0, feat_b=1):
#         z_torus = Z_numpy[:, z_dim_euc:]
#         u1, v1 = z_torus[:, 2*feat_a], z_torus[:, 2*feat_a+1]
#         u2, v2 = z_torus[:, 2*feat_b], z_torus[:, 2*feat_b+1]
        
#         theta = np.arctan2(v1, u1)
#         phi   = np.arctan2(v2, u2)
        
#         R, r = 3, 1
#         x = (R + r*np.cos(theta))*np.cos(phi)
#         y = (R + r*np.cos(theta))*np.sin(phi)
#         z = r*np.sin(theta)
        
#         fig = plt.figure(figsize=(10,7))
#         ax = fig.add_subplot(111, projection='3d')
#         sc = ax.scatter(x, y, z, c=theta, cmap='twilight', s=5, alpha=0.6)
#         ax.set_axis_off()
#         ax.set_xlim(-4,4); ax.set_ylim(-4,4); ax.set_zlim(-4,4)
#         plt.title(f"Torus latent (feat {feat_a} & {feat_b})")
#         plt.show()

#     @staticmethod
#     def plot_torus_latent(Z_train, num_cyc_features, z_dim_euclid):
#         fig, axes = plt.subplots(1, num_cyc_features, figsize=(num_cyc_features * 4, 4))
#         if num_cyc_features == 1: axes = [axes]
        
#         for i in range(num_cyc_features):
#             # Indexing: Euclid uses first z_dim_euclid dims. 
#             # Torus follows in pairs of 2.
#             idx = z_dim_euclid + (i * 2)
#             x = Z_train[:, idx]     # cos
#             y = Z_train[:, idx + 1] # sin
            
#             axes[i].scatter(x, y, s=2, alpha=0.5)
#             axes[i].set_aspect('equal')
#             axes[i].set_title(f"Circle {i} (Latent Space)")
#             axes[i].set_xlim(-1.1, 1.1); axes[i].set_ylim(-1.1, 1.1)
#         plt.show()

#     @staticmethod
#     def plot_kappa_dist(enc_t, loader, device):
#         enc_t.eval()
#         all_kappas = []
#         with torch.no_grad():
#             for _, b_cyc in loader:
#                 _, kappa = enc_t(b_cyc.to(device))
#                 all_kappas.append(kappa.cpu())
#         kappas = torch.cat(all_kappas).numpy()
#         plt.hist(kappas, bins=50)
#         plt.title("Distribution of Concentration (Kappa)")
#         plt.show()


# PlotTopolin.plot_kappa_dist(enc_t, loader, device)

rmse_torlin, r2_torlin, mae_torlin = [], [], []
rmse_oldtor, r2_oldtor, mae_oldtor = [], [], []
for i in range(1):
    rmse_i, r2_i, mae_i, (Z_train, Z_test), (y_hat, y_train_w, y_test_w), (z_dim_euclid, z_dim_torus) = Torlin.run_torlin(X_train,
                                                                X_test, y_train, y_test, p, angular_col_names_list, epsilon=1e-3)
    # rmse_i, r2_i, mae_i, (Z_train, Z_test) = Torlin.run_sphetorlin_pipeline(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    #                                                                         p=p, angular_features_list=angular_col_names_list)
    rmse_torlin.append(rmse_i)
    r2_torlin.append(r2_i)
    mae_torlin.append(mae_i)

    # rmse_oldtor_i, r2_oldtor_i, mae_oldtor_i, (Z_train, Z_test) = run_oldtor_LSTM(X_train, X_test, y_train, y_test, p, epsilon=epsilon)
    # rmse_oldtor.append(rmse_oldtor_i)
    # r2_oldtor.append(r2_oldtor_i)
    # mae_oldtor.append(mae_oldtor_i)

print(f"Torlin RMSE={np.mean(rmse_torlin):.3f}±{np.std(rmse_torlin):.3f} \
        MAE={np.mean(mae_torlin):.3f}±{np.std(mae_torlin):.3f} \
        R2={np.mean(r2_torlin):.3f}±{np.std(r2_torlin):.3f}")

# print(f"oldtor RMSE={np.mean(rmse_oldtor):.3f}±{np.std(rmse_oldtor):.3f} \
#         MAE={np.mean(mae_oldtor):.3f}±{np.std(mae_oldtor):.3f} \
#         R2={np.mean(r2_oldtor):.3f}±{np.std(r2_oldtor):.3f}")

# rmse_torlin, r2_torlin, mae_torlin, (Z_train, Z_test) = run_torlin_pipeline(X_train, X_test, y_train, y_test, p)
# print(f"Torlin VAE, RMSE={rmse_torlin:.3f}  MAE={mae_torlin:.3f} R2={r2_torlin:.3f}")

# rmse_oldtor, r2_oldtor, mae_oldtor, (Z_train, Z_test) = run_oldtor_LSTM(X_train, X_test, y_train, y_test, p)
# print(f"OldTor VAE, RMSE={rmse_oldtor:.3f}  MAE={mae_oldtor:.3f} R2={r2_oldtor:.3f}")

_, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose=False)
# plot_torus(Z_train, z_dim_euc=p.z_dim_total - 2*X_cyc_train.shape[1])

# plot_publication_torus(Z_test, z_dim_euc=z_dim_euclid, feat_a=0, feat_b=1)
# plot_publication_torus(Z_test, z_dim_euc=z_dim_euclid, feat_a=2, feat_b=3)

TopolinPlots.plot_torus_latent(Z_train, len(angular_col_names_list), p.z_dim_total - 2*len(angular_col_names_list))


In [ ]:
"California housing dataset (tabular) -  has latitude/longitude info"

dataset = "cali_housing"

from sklearn.datasets import fetch_california_housing

# as_frame=True returns a Pandas DataFrame immediately
data = fetch_california_housing(as_frame=True)
df = data.frame

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"].values[:, None] # made to 2D array


In [ ]:
"🚫 (cyclic NOT periodic, period.= 3) 🚲 Bike sharing dataset (https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)"

dataset = "bike_sharing"

bike_sharing = fetch_ucirepo(id=275) 
  
X = bike_sharing.data.features 
y = bike_sharing.data.targets
# print(bike_sharing.variables) 

X = X.drop(["dteday"], axis=1)


In [ ]:
"🚫 [not good] Forest fires (https://archive.ics.uci.edu/dataset/162/forest+fires)"
  
dataset = "forest_fires"

forest_fires = fetch_ucirepo(id=162) 

# data (as pandas dataframes) 
X = forest_fires.data.features 
y = forest_fires.data.targets 

# print(forest_fires.variables)

# month: jan=0, feb=1, ..., dec=11
month_map = {m: i for i, m in enumerate(['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'])}
day_map   = {d: i for i, d in enumerate(['mon','tue','wed','thu','fri','sat','sun'])}

X = X.copy()
X.loc[:, 'month'] = X['month'].map(month_map)
X.loc[:, 'day']   = X['day'].map(day_map)



In [ ]:
"🚫👤 activity (https://archive.ics.uci.edu/dataset/341/smartphone+based+recognition+of+human+activities+and+postural+transitions)"

dataset = "activity"

dataset_loc = "../public_datasets/2D/tabular/activity_recognition/"
subject_id_train_file = "subject_id_train.txt"
subject_id_test_file  = "subject_id_test.txt"

train_folder = "Train/"
test_folder  = "Test/"

y_test_file  = "y_test.txt"
y_train_file = "y_train.txt"

X_train_file = "X_train.txt"
X_test_file  = "X_test.txt"

X_train_loc = f"{dataset_loc}/{train_folder}/{X_train_file}"
y_train_loc = f"{dataset_loc}/{train_folder}/{y_train_file}"
subject_id_train_loc = f"{dataset_loc}/{train_folder}/{subject_id_train_file}"

X_test_loc = f"{dataset_loc}/{test_folder}/{X_test_file}"
y_test_loc = f"{dataset_loc}/{test_folder}/{y_test_file}"
subject_id_test_loc = f"{dataset_loc}/{test_folder}/{subject_id_test_file}"


X_train = pd.read_csv(X_train_loc, sep='\s+', header=None)
y_train = pd.read_csv(y_train_loc, sep='\s+', header=None)


for i in range(X.shape[1]):
    cycle_score = compute_cyclicity_score(X.iloc[:,i].values)
    print(f"Cyclicity score of feature {i}: {cycle_score}")

X_train.head()


In [ ]:
"🚫🥙 IMU gyro (https://www.kaggle.com/datasets/banaankiamanesh/imu-for-ai?select=IMU_Data_1.csv)"
# euler angles = pitch/yaw/roll
# euler angles are not cyclic (locked at certain angles)
# periodicity is coordinate-dependept
# so forcing the angles onto a circle is a fake wraparound

from scipy.signal import butter, filtfilt

def apply_lowpass_filter(df, cutoff=5.0, fs=100.0, order=4):
    """
    Applies a zero-phase Butterworth low-pass filter to all columns.
    df: Pandas DataFrame
    cutoff: Cutoff frequency in Hz
    fs: Sampling frequency in Hz
    order: Polynomial order of the filter
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    
    df_filtered = df.copy()
    for col in df.columns:
        # filtfilt applies the filter twice (forward and backward) to eliminate phase shift
        df_filtered[col] = filtfilt(b, a, df[col].values)
    
    return df_filtered

dataset  = "imu_gyro"
file_loc = "../public_datasets/2D/tabular/imu_data/IMU_Data_1.csv"

X      = pd.read_csv(file_loc)
y_cols = ["Quat_0","Quat_1","Quat_2","Quat_3"]

y = X[y_cols].values
X = X.drop(columns=y_cols)

# 2. Filter X (Exclude non-numeric columns if any exist)
X_filtered = apply_lowpass_filter(X, cutoff=5.0, fs=100.0)
X = X_filtered

for col in range(X.shape[1]):
    data_col    = X.iloc[:, col].to_numpy().flatten()
    cycle_score = compute_cyclicity_score(data_col)
    print(f"col {col} cycl. score: {cycle_score:.4f}")


In [ ]:
"🧭PAMAP"

dataset = "PAMAP"

dataset_loc = "../public_datasets/2D/tabular/PAMAP2"
record_name = "Protocol/subject108.dat"
record_path = f"{dataset_loc}/{record_name}"
df          = pd.read_csv(record_path, sep=' ', header=None)

# step 1: preprocess
df = df.bfill().ffill()  # fill NaNs for all columns immediately
df = df.sample(frac=0.9, random_state=42) # reduce

y  = df.iloc[:, 1].values
X  = df.drop(columns=[1]) # drop timestamp and subject ID and activity ID


for i in range(X.shape[1]):
    cycle_score = compute_cyclicity_score(X.iloc[:,i].values)
    print(f"Cyclicity score of feature {i}: {cycle_score}")


In [ ]:
"another weather (https://www.kaggle.com/datasets/muthuj7/weather-dataset?select=weatherHistory.csv)"

df = pd.read_csv('../public_datasets/2D/tabular/another_weather/weatherHistory.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")


In [ ]:
"cities weather (https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data?resource=download)"

weather_folder   = "../public_datasets/2D/tabular/Hourly Weather Data 2012-2017"
files_to_combine = ["temperature.csv", "humidity.csv", "wind_speed.csv", "pressure.csv", "wind_direction.csv"]

dfs = {f.split(".")[0]: pd.read_csv(os.path.join(weather_folder, f)) for f in files_to_combine}

# Extract city names (assumes all files have same columns)
cities     = [col for col in dfs["temperature"].columns if col != "datetime"]
timesteps  = len(dfs["temperature"])
properties = len(files_to_combine)

weather_array = np.zeros((len(cities), timesteps, properties), dtype=float)

# Fill array
for p, prop in enumerate(files_to_combine):
    df_prop = dfs[prop.split(".")[0]]  # remove .csv from name
    for c, city in enumerate(cities):
        weather_array[c, :, p] = df_prop[city].values

for c in range(weather_array.shape[0]):        # cities
    for p in range(weather_array.shape[2]):    # properties
        col = weather_array[c, :, p]
        if np.isnan(col).any():
            mean_val = np.nanmean(col)  # compute mean ignoring NaNs
            col[np.isnan(col)] = mean_val
            weather_array[c, :, p] = col
print("Array shape:", weather_array.shape)
print("NaNs remaining:", np.isnan(weather_array).sum())

cycle_score = compute_cyclicity_score(weather_array[0, :, 0])
print(f"Cyclicity score : {cycle_score}")



In [ ]:
"Pseudo-cyclic synthetic dataset (https://archive.ics.uci.edu/dataset/136/pseudo+periodic+synthetic+time+series)"

data = np.loadtxt('../public_datasets/2D/tabular/pseudo_cyclic/synthetic.data')

for i in range(data.shape[1]):
    cycle_score = compute_cyclicity_score(data[:,i])
    print(f"Cyclicity score of feature {i}: {cycle_score}")

In [ ]:
"Traffic flow dataset (https://archive.ics.uci.edu/dataset/608/traffic+flow+forecasting)"

from scipy.io import loadmat

# Load data
data = loadmat("../public_datasets/2D/tabular/traffic_dataset/traffic_dataset.mat")

def flatten_X(mat_array):
    """
    Convert 1xN MATLAB object array of 36x48 matrices into 2D numeric array
    N rows, 36*48 columns
    """
    flattened = []
    for m in mat_array[0]:
        # Ensure numeric type
        flattened.append(np.array(m, dtype=float).reshape(-1))
    return np.stack(flattened, axis=0)

# Flatten input features
X_train_np = flatten_X(data['tra_X_tr'])
X_test_np  = flatten_X(data['tra_X_te'])

# Outputs: already numeric, just transpose to N x 36
y_train_np = data['tra_Y_tr'].T.astype(float)
y_test_np  = data['tra_Y_te'].T.astype(float)

# Convert to Polars
X_train = pl.DataFrame(X_train_np)
X_test  = pl.DataFrame(X_test_np)
y_train = pl.DataFrame(y_train_np)
y_test  = pl.DataFrame(y_test_np)

print(X_train.shape, y_train.shape)
print(X_train.head())
print(y_train.head())



In [ ]:
"🧮 periodogram-based method"

def periodogram_analysis(df, fs=10):
    # fs is sampling frequency. If 1 sample = 1 day, fs=1.
    cycle_counts = []
    
    for col in df.columns:
        f, Pxx_den = periodogram(df[col].values, fs)
        
        # Find the frequency with the highest power (excluding DC/0 frequency)
        idx        = np.argmax(Pxx_den[1:]) + 1 
        peak_freq  = f[idx]
        
        if Pxx_den[idx] > (Pxx_den.mean() * 5): # Simple threshold for "significant"
            print(f"Feature '{col}' has a strong cycle at freq {peak_freq:.5f}")
            cycle_counts.append(peak_freq)
            
    # Count unique frequencies to find 'k'
    # Use rounding because frequencies might be slightly off due to noise
    unique_freqs = np.unique(np.round(cycle_counts, 4))
    print(f"\nSuggested # of latent circles (k): {len(unique_freqs)}")

    dominant_freq = np.max(cycle_counts)
    return len(unique_freqs), dominant_freq

# def _dominant_periods(x: np.ndarray, fs: float = 1.0, topk: int = 2, max_period: int = None) -> list[int]:
#     """Return top-k dominant periods, ignoring near-DC artifacts."""
#     n       = len(x)
#     freqs   = np.fft.rfftfreq(n, d=1/fs)[1:]
#     power   = np.abs(np.fft.rfft(x))[1:]**2
#     periods = 1 / freqs

#     if max_period is None:
#         max_period = int(n * 0.25)

#     mask = periods <= max_period
#     idxs = np.argsort(power[mask])[-topk:]
#     return sorted(int(round(periods[mask][i])) for i in idxs)

def dataset_dominant_period(df: pd.DataFrame, fs: float = 1.0, topk: int = 1, max_period: int = None, cyclic_threshold: float = 5.0):
    periods = []
    for col in df.columns:
        if not np.issubdtype(df[col].dtype, np.number): continue
        x      = df[col].values
        f, Pxx = periodogram(x, fs)
        if Pxx[1:].max() <= cyclic_threshold * Pxx.mean(): continue
        p      = Periodicity._dominant_periods(x, fs=fs, topk=topk, max_period=max_period)[0]
        periods.append(p)

    if not periods: return None, None
    mode_period = int(pd.Series(periods).mode().iloc[0])
    return mode_period, fs / mode_period

def dataset_dominant_period_weighted(X: pd.DataFrame, y, fs: float = 1.0) -> tuple[int, float]:
    """Return weighted dominant period using spectral power × corr with target."""
    y_values = np.asarray(y).ravel()
    period_votes = {}

    for col in X.columns:
        if not np.issubdtype(X[col].dtype, np.number): continue
        x = X[col].values

        corr = np.abs(np.corrcoef(x, y_values)[0, 1])
        if np.isnan(corr): continue

        f, Pxx = periodogram(x, fs)
        idx = np.argmax(Pxx[1:]) + 1
        period = int(round(fs / f[idx]))
        period_votes[period] = period_votes.get(period, 0) + Pxx[idx] * corr

    if not period_votes:
        return 24, fs / 24

    best = max(period_votes, key=period_votes.get)
    return best, fs / best

def seasonal_decompose(x: np.ndarray, P: int) -> tuple[np.ndarray, np.ndarray]:
    t     = np.arange(len(x))
    theta = 2 * np.pi * t / P
    C     = np.stack([np.cos(theta), np.sin(theta)], axis=1)
    a, b  = np.linalg.lstsq(C, x, rcond=None)[0]
    s     = a*C[:,0] + b*C[:,1]
    r     = x - s
    return s, r

def multi_phase_encode_window(window_len: int, periods: list[int]) -> np.ndarray:
    t = np.arange(window_len)
    out = []
    for P in periods:
        theta = 2*np.pi*t/P
        out.append(np.stack([np.cos(theta), np.sin(theta)], axis=1))
    return np.concatenate(out, axis=1)

def decompose_periodic_features(df: pd.DataFrame, fs: float = 1.0, cyclic_threshold: float = 5.0, topk: int = 2, max_frac: float = 0.25) -> tuple[pd.DataFrame, list[str]]:
    """Decompose periodic features and add sin/cos phase encodings."""
    residual_df = df.copy()
    phase_cols: list[str] = []

    for col in df.columns:
        if not np.issubdtype(df[col].dtype, np.number): continue

        x = df[col].values
        f, Pxx = periodogram(x, fs)
        if Pxx[1:].max() <= cyclic_threshold * Pxx.mean(): continue

        periods = Periodicity._dominant_periods(x, fs=fs, topk=topk, max_period=int(len(x)*max_frac) if max_frac is not None else None)
        main_period = periods[0]
        freq = fs / main_period
        print(f"Col '{col}' dominant period={main_period}, freq={freq:.6f}")

        s, _ = seasonal_decompose(x, main_period)
        residual_df[col] = x - s

        t = np.arange(len(x))
        for i, P in enumerate(periods):
            theta = 2 * np.pi * t / P
            residual_df[f"{col}_cos_{i}"] = np.cos(theta)
            residual_df[f"{col}_sin_{i}"] = np.sin(theta)
            phase_cols += [f"{col}_cos_{i}", f"{col}_sin_{i}"]
    return residual_df, phase_cols

# --- 2. MIXED DENSITY & TC FUNCTIONS ---
def log_density_mixed(z_combined, mu_e, logvar_e, mu_t, kappa_t):
    batch_size  = z_combined.size(0)
    z_dim_e     = mu_e.size(1)
    num_circles = mu_t.size(1)

    z_e = z_combined[:, :z_dim_e].unsqueeze(1)
    m_e = mu_e.unsqueeze(0)
    v_e = logvar_e.unsqueeze(0)
    log_q_ze = -0.5 * (np.log(2 * np.pi) + v_e + (z_e - m_e)**2 * torch.exp(-v_e)).sum(dim=-1)
    
    z_t = z_combined[:, z_dim_e:].view(batch_size, num_circles, 2).unsqueeze(1)
    m_t = mu_t.unsqueeze(0)
    k_t = kappa_t.unsqueeze(0)
    inner_prod = (z_t * m_t).sum(dim=-1)
    log_norm_t = torch.log(2 * np.pi * torch.i0(k_t.squeeze(-1) + 1e-6))
    log_q_zt   = (k_t.squeeze(-1) * inner_prod - log_norm_t).sum(dim=-1)
    return log_q_ze + log_q_zt

def calculate_mixed_tc(z, mu_e, logvar_e, mu_t, kappa_t):
    log_qz       = log_density_mixed(z, mu_e, logvar_e, mu_t, kappa_t)
    log_qz_joint = torch.logsumexp(log_qz, dim=1) - np.log(z.size(0))
    log_qz_marg  = log_qz.mean(dim=1)
    return (log_qz_joint - log_qz_marg).mean()


# --- 2. UPDATED PIPELINE FUNCTION ---
def run_beta_tc_torlin_pipeline(X_lin_tr, X_lin_te, X_cyc_tr, X_cyc_te, y_train, y_test, p, beta=2.0, k_circles=4):
    # Scaling
    X_lin_tr_s, X_lin_te_s = scale_train_and_test_sets(X_lin_tr, X_lin_te)
    X_cyc_tr_s, X_cyc_te_s = scale_train_and_test_sets(X_cyc_tr, X_cyc_te)
    
    # Windowing
    W_lin_tr = Windowing.make_windows_from_X(X_lin_tr_s, p.window_size, sliding_size, horizon=p.horizon).to(device)
    W_cyc_tr = Windowing.make_windows_from_X(X_cyc_tr_s, p.window_size, sliding_size, horizon=p.horizon).to(device)
    W_lin_te = Windowing.make_windows_from_X(X_lin_te_s, p.window_size, sliding_size, horizon=p.horizon).to(device)
    W_cyc_te = Windowing.make_windows_from_X(X_cyc_te_s, p.window_size, sliding_size, horizon=p.horizon).to(device)

    # Latent Config
    z_dim_spheric = k_circles * 2
    z_dim_euclid  = max(12, p.z_dim_total - z_dim_spheric)
    
    # Models
    h_split = int(p.hidden_dim / np.sqrt(2))
    enc_e   = LSTMEncoderEuclid(X_lin_tr.shape[1], h_split, z_dim_euclid).to(device)
    enc_t   = LSTMToroidalEncoder(X_cyc_tr.shape[1], h_split, k_circles).to(device)
    decoder = MLPDecoder(z_dim_euclid + z_dim_spheric, p.window_size, 
                         X_lin_tr.shape[1] + X_cyc_tr.shape[1], p.hidden_dim).to(device)
    
    optimizer = torch.optim.AdamW(list(enc_e.parameters()) + list(enc_t.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)
    loader    = DataLoader(TensorDataset(W_lin_tr, W_cyc_tr), batch_size=p.batch_size, shuffle=True)

    # Training
    for epoch in range(p.epochs):
        enc_e.train(); enc_t.train(); decoder.train()
        for b_lin, b_cyc in loader:
            optimizer.zero_grad()
            mu_e, lv_e = enc_e(b_lin); z_e = Reparam.reparam_gaussian(mu_e, lv_e)
            # mu_t, kp_t = enc_t(b_cyc); z_t = torch.cat([Torlin.sample_vmf_approximate(mu_t[:, i, :], kp_t[:, i]) for i in range(k_circles)], dim=-1)
            mu_t, kp_t = enc_t(b_cyc); z_t = torch.cat([Torlin.sample_vmf_exact(mu_t[:, i, :], kp_t[:, i]) for i in range(k_circles)], dim=-1)
            
            z_joint = torch.cat([z_e, z_t], dim=-1)
            recon   = decoder(z_joint)
            
            target     = torch.cat([b_lin, b_cyc], dim=-1).view(b_lin.size(0), -1)
            loss_recon = F.mse_loss(recon, target)
            
            tc_loss = calculate_mixed_tc(z_joint, mu_e, lv_e, mu_t, kp_t)
            kl_e    = -0.5 * torch.sum(1 + lv_e - mu_e.pow(2) - lv_e.exp(), dim=1).mean()
            kl_t    = sum((kp_t[:, i] - torch.log(kp_t[:, i] + 1e-6)).mean() for i in range(k_circles))
            loss    = p.lambda_recon * loss_recon + kl_e + kl_t + (beta - 1.0) * tc_loss
            loss.backward(); optimizer.step()

    # Evaluation
    enc_e.eval(); enc_t.eval()
    with torch.no_grad():
        def encode(l, c):
            me, _ = enc_e(l); mt, _ = enc_t(c)
            zt    = torch.cat([F.normalize(mt[:, i, :], dim=-1) for i in range(k_circles)], dim=-1)
            return torch.cat([me, zt], dim=-1).cpu().numpy()
        Z_tr, Z_te = encode(W_lin_tr, W_cyc_tr), encode(W_lin_te, W_cyc_te)

    y_tr_s, y_te_s = scale_train_and_test_sets(y_train, y_test)
    y_tr_win = Windowing.make_windows_from_y(y_tr_s, p.window_size, sliding_size, task=p.task, horizon=p.horizon)
    y_te_win = Windowing.make_windows_from_y(y_te_s, p.window_size, sliding_size, task=p.task, horizon=p.horizon)
    
    y_hat = fit_catboost_multi(Z_tr, y_tr_win, Z_te)
    mae   = mean_absolute_error(y_te_win, y_hat)
    return root_mean_squared_error(y_te_win, y_hat), r2_score(y_te_win, y_hat), mae, (Z_tr, Z_te)

# Run decomposition (Fixes the discontinuous theta jump)
X_train_new, phase_cols = decompose_periodic_features(X_train, cyclic_threshold=5.0)
X_test_new, _           = decompose_periodic_features(X_test, cyclic_threshold=5.0)

print(dataset)

mode_period, mode_freq = dataset_dominant_period(X_train, fs=1.0, max_period=365)
print(f"Dataset dominant period: {mode_period}, frequency: {mode_freq}")

mode_period, mode_freq = dataset_dominant_period_weighted(X_train_new, y_train, fs=1.0)
print(f"Weighted dominant period: {mode_period}, frequency: {mode_freq}")

# Split
X_lin_tr = X_train_new.drop(columns=phase_cols)
X_lin_te = X_test_new.drop(columns=phase_cols)
X_cyc_tr = X_train_new[phase_cols]
X_cyc_te = X_test_new[phase_cols]


rmse_periodogram, r2_periodogram, mae_periodogram = [], [], []
for i in range(5):
    rmse_i, r2_i, mae_i, (Z_train, Z_test) = run_beta_tc_torlin_pipeline(X_lin_tr, X_lin_te, X_cyc_tr, X_cyc_te, y_train, y_test, p=p, beta=2, k_circles=8)
    rmse_periodogram.append(rmse_i)
    r2_periodogram.append(r2_i)
    mae_periodogram.append(mae_i)

print(f"Periodogram RMSE={np.mean(rmse_periodogram):.4f}±{np.std(rmse_periodogram):.4f} \
        MAE={np.mean(mae_periodogram):.4f}±{np.std(mae_periodogram):.4f} \
        R2={np.mean(r2_periodogram):.4f}±{np.std(r2_periodogram):.4f}")


In [ ]:
# plt.figure(figsize=(15,5))
# plt.plot(y[:2_000])
# plt.show()

dom_period = Periodicity._dominant_periods(y, fs=1.0, topk=1, max_period=400)[0]
print(f"Dominant period of y: {dom_period}")

def x_dominant_periods(x: np.ndarray, fs: float = 1.0, topk: int = 2, max_period: int = None) -> list[int]:
    """Return top-k dominant periods, ignoring near-DC artifacts."""
    n       = len(x)
    freqs   = np.fft.rfftfreq(n, d=1/fs)[1:]
    power   = np.abs(np.fft.rfft(x))[1:]**2
    periods = 1 / freqs

    if max_period is None:
        max_period = int(n * 0.25)

    mask = periods <= max_period
    idxs = np.argsort(power[mask])[-topk:]
    return sorted(int(round(periods[mask][i])) for i in idxs)



In [ ]:
z_detached = Z_train.detach().cpu().numpy()

z_s = 12 # num of latent dims

plt.figure(figsize=(15,12))
# for col in range(z_detached.shape[1]):
    # plt.plot(z_detached[:, col], label=f"dim {col}", alpha=0.7)
    # plt.plot(z_detached[:, -z_s:])
plt.plot(z_detached[:, -z_s:])

plt.title("Latent Dimensions over Time")
plt.xlabel("Window Index")
plt.ylabel("Latent Value")
plt.legend()

z_var = z_detached.var(axis=0)
# print(z_var[:], z_var.mean())
print((z_var < 1e-4).mean())


In [ ]:

z_detached = Z_train.detach().cpu().numpy()   # shape (T, D)
y = y_train_w.mean(axis=1)           # shape (T,)
t = np.arange(len(z_detached))

# phase
# omega   = 2 * np.pi / dominant_period
# phase_i = max(abs(np.corrcoef(z_detached[:, i], np.sin(omega*t))[0,1]), abs(np.corrcoef(z_detached[:, i], np.cos(omega*t))[0,1]))

# amplitude: z magnitude correlates with envelope of signal, Non-zero variance but non-cyclic, Correlates with |FFT amplitude| or rolling std of x
def rolling_std(x, win=200):
    """Compute rolling std using a box filter of size `win` and the global mean."""
    return np.sqrt(np.convolve((x - x.mean())**2, np.ones(win)/win, mode="same"))

amplitude_coeff = np.corrcoef(z_detached[:, i], rolling_std(y))[0,1]

# trend: Low-frequency monotonic movement + High corr with time or rolling mean
trend_coeff = np.corrcoef(z_detached[:, i], t)[0,1]
trend_i     = abs(np.corrcoef(z_detached[:, i], t)[0,1])

# variance (collapse detector)
var_i = z_detached[:, i].var()

# diagnostic table
rows = []
for i in range(z_detached.shape[1]):
    rows.append({
        "dim": i,
        "var": z_detached[:, i].var(),
        # "phase": phase_i,
        "trend": trend_i,
        "amp": amplitude_coeff,
        "var": var_i,})
df = pd.DataFrame(rows)

print(df)


In [ ]:
"☑️ 100% EUCLIDEAN / Spherical (NO SPLIT)"

# ====== euclidean ======
def run_euclidean_vae_scenario_MLP(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                   y_test: np.ndarray, p: RunParams,) -> Tuple[float, float, float]:
    """Run Euclidean-only VAE baseline. Returns (rmse, r2)."""

    # ---- scale (match split-VAE MLP) ----
    X_train, X_test = scale_train_and_test_sets(X_train, X_test)
    y_train, y_test = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train, p.window_size, sliding_size, horizon=p.horizon).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test, p.window_size, sliding_size, horizon=p.horizon).to(device)

    X_train_w = X_train_w.view(-1, p.window_size, X_train.shape[-1]).to(device)
    X_test_w  = X_test_w.view(-1, p.window_size, X_test.shape[-1]).to(device)

    # ---- model ----
    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
    encoder   = EuclidEncoder(window_size=p.window_size, input_dim=X_train.shape[1], z_dim=p.z_dim_total, hidden=hidden_dim_split).to(device)
    decoder   = Decoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden=hidden_dim_split).to(device)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer,)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False,)

    lambdas = {
        "reconstr": p.lambda_recon,
        "euc": p.lambda_kl_euc / np.sqrt(p.z_dim_total),
        "sph": 0.0,}

    best_loss = float("inf")
    counter   = 0
    for _ in range(p.epochs):
        epoch_loss = NoSplit.train_vae_no_split(train_loader, encoder, decoder, optimizer, lambdas)
        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode ----
    Z_train = NoSplit.encode_dataset_no_split(X_train_w, encoder, pooling=None)
    Z_test  = NoSplit.encode_dataset_no_split(X_test_w, encoder, pooling=None)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, sliding_size, task=p.task, horizon=p.horizon)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, sliding_size, task=p.task, horizon=p.horizon)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    mae   = mean_absolute_error(y_test_win, y_hat)
    return rmse, r2, mae

def run_euclidean_vae_scenario_LSTM(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                    y_test: np.ndarray, p: RunParams) -> tuple[float, float, float]:
    """Run Euclidean VAE with LSTM encoder and MLP decoder."""

    # ---- scale + window ----
    X_train_t, X_test_t           = scale_train_and_test_sets(X_train, X_test)
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)
    X_train_w = Windowing.make_windows_from_X(X_train_t, p.window_size, sliding_size, horizon=p.horizon).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test_t, p.window_size, sliding_size, horizon=p.horizon).to(device)

    # ---- models ----
    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
    encoder   = LSTMEncoderEuclid(input_dim=X_train.shape[1], hidden_dim=hidden_dim_split, z_dim=p.z_dim_total).to(device)
    decoder   = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=p.hidden_dim,).to(device)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer,)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False,)

    lambdas = {"reconstr": p.lambda_recon, "euc": p.lambda_kl_euc / np.sqrt(p.z_dim_total),}

    best_loss = float("inf")
    counter   = 0
    for _ in range(p.epochs):
        epoch_loss = 0.0
        for (x,) in train_loader:
            optimizer.zero_grad()

            mu, logvar = encoder(x)
            z          = Reparam.reparam_gaussian(mu, logvar)
            x_hat_flat = decoder(z)
            x_hat      = x_hat_flat.view_as(x)

            L_rec = F.mse_loss(x_hat, x)
            L_kl  = kl_gaussian(mu, logvar)
            loss  = lambdas["reconstr"] * L_rec + lambdas["euc"] * L_kl
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode (NO POOLING) ----
    encoder.eval()
    with torch.no_grad():
        mu_train, _ = encoder(X_train_w)
        mu_test,  _ = encoder(X_test_w)

    Z_train = mu_train
    Z_test  = mu_test

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, sliding_size, task=p.task, horizon=p.horizon)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, sliding_size, task=p.task, horizon=p.horizon)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    # y_hat = LinearRegression().fit(Z_train.cpu().numpy(), y_train_win).predict(Z_test.cpu().numpy())
    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    mae   = mean_absolute_error(y_test_win, y_hat)
    return rmse, r2, mae


# ====== spherical ======
def run_spherical_vae_scenario_MLP(X_train: pd.DataFrame,X_test: pd.DataFrame, y_train: np.ndarray,
                                   y_test: np.ndarray, p: RunParams,) -> tuple[float, float, float]:
    """Run 100% spherical VAE baseline. Returns (rmse, r2)."""

    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))

    # ---- scale (match Euclidean baseline) ----
    X_train, X_test = scale_train_and_test_sets(X_train, X_test)
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train, p.window_size, sliding_size, horizon=p.horizon).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test, p.window_size, sliding_size, horizon=p.horizon).to(device)

    X_train_w = X_train_w.view(-1, p.window_size, X_train.shape[-1]).to(device)
    X_test_w  = X_test_w.view(-1, p.window_size, X_test.shape[-1]).to(device)

    # ---- model ----
    encoder  = SphericalEncoder(window_size=p.window_size, input_dim=X_train.shape[1], z_dim=p.z_dim_total, hidden=hidden_dim_split).to(device)
    decoder  = Decoder(z_dim_total=p.z_dim_total,window_size=p.window_size, output_dim=X_train.shape[1],hidden=hidden_dim_split).to(device)
    optimizer= torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()),lr=p.lr_optimizer,)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False)

    lambdas = {
        "reconstr": p.lambda_recon,
        "sph": p.lambda_kl_sph,
        "euc": 0.0,}

    best_loss = float("inf")
    counter   = 0

    for _ in range(p.epochs):
        epoch_loss = 0.0
        encoder.train()
        decoder.train()

        for (x,) in train_loader:
            optimizer.zero_grad()

            B, win, D = x.shape
            x_flat    = x.view(B, win * D)
            mu, kappa = encoder(x_flat)
            z         = Reparam.reparam_vmf(mu, kappa)

            x_hat     = decoder(z)
            L_rec     = F.mse_loss(x_hat, x_flat)
            L_sph     = regularization_vmf(kappa, z.size(-1))

            loss = lambdas["reconstr"] * L_rec + lambdas["sph"] * L_sph
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode (NO EUCLIDEAN MEAN) ----
    encoder.eval()
    with torch.no_grad():
        mu_train, _ = encoder(X_train_w.view(X_train_w.size(0), -1))
        mu_test,  _ = encoder(X_test_w.view(X_test_w.size(0), -1))

    Z_train = F.normalize(mu_train, dim=-1)
    Z_test  = F.normalize(mu_test,  dim=-1)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, sliding_size, task=p.task, horizon=p.horizon)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, sliding_size, task=p.task, horizon=p.horizon)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    return rmse, r2

def run_spherical_vae_scenario_LSTM(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                                    y_test: np.ndarray, p: RunParams,) -> tuple[float, float, float]:
    """Run 100% spherical VAE with LSTM encoder and MLP decoder for timeseries."""

    # ---- scale ----
    X_train_scaled, X_test_scaled = scale_train_and_test_sets(X_train, X_test)
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train_scaled, p.window_size, sliding_size, horizon=p.horizon).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test_scaled,  p.window_size, sliding_size, horizon=p.horizon).to(device)

    # ---- reshape for LSTM: (B, T, D) ----
    X_train_w = X_train_w.view(-1, p.window_size, X_train.shape[-1]).to(device)
    X_test_w  = X_test_w.view(-1, p.window_size, X_test.shape[-1]).to(device)

    # ---- model ----
    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
    encoder   = LSTMSphericalEncoder(input_dim=X_train.shape[1], hidden_dim=hidden_dim_split, z_dim=p.z_dim_total).to(device)
    decoder   = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=p.hidden_dim,).to(device)
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)

    # ---- training ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=False)
    lambdas      = {"reconstr": p.lambda_recon, "sph": p.lambda_kl_sph, "euc": 0.0}

    best_loss = float("inf")
    counter   = 0

    for _ in range(p.epochs):
        epoch_loss = 0.0
        encoder.train()
        decoder.train()

        for (x,) in train_loader:
            optimizer.zero_grad()

            mu, kappa = encoder(x)              # LSTM encoder
            z         = Reparam.reparam_vmf(mu, kappa)

            x_hat_flat = decoder(z)
            x_hat      = x_hat_flat.view_as(x)

            L_rec = F.mse_loss(x_hat, x)
            L_sph = regularization_vmf(kappa, z.size(-1))
            loss  = lambdas["reconstr"] * L_rec + lambdas["sph"] * L_sph
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        best_loss, counter, stop = early_stop(epoch_loss, best_loss, counter, p.earlystop_patience)
        if stop:
            break

    # ---- encode (no Euclidean mean) ----
    encoder.eval()
    with torch.no_grad():
        mu_train, _ = encoder(X_train_w)
        mu_test,  _ = encoder(X_test_w)

    Z_train = F.normalize(mu_train, dim=-1)
    Z_test  = F.normalize(mu_test,  dim=-1)

    # ---- regression ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task, horizon=p.horizon)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled,  p.window_size, p.sliding_size, task=p.task, horizon=p.horizon)

    assert Z_train.shape[0] == y_train_win.shape[0]
    assert Z_test.shape[0]  == y_test_win.shape[0]

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy(),)
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)
    mae   = mean_absolute_error(y_test_win, y_hat)
    return rmse, r2, mae

rmse_euclid, r2_euclid, mae_euclid          = [], [], []
rmse_spherical, r2_spherical, mae_spherical = [], [], []
for i in range(5):
    rmse_euclid_i, r2_euclid_i, mae_euclid_i = run_euclidean_vae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
    rmse_euclid.append(rmse_euclid_i)
    r2_euclid.append(r2_euclid_i)
    mae_euclid.append(mae_euclid_i)

    rmse_spherical_i, r2_spherical_i, mae_spherical_i = run_spherical_vae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
    rmse_spherical.append(rmse_spherical_i)
    r2_spherical.append(r2_spherical_i)
    mae_spherical.append(mae_spherical_i)

print(f"Euclidean RMSE={np.mean(rmse_euclid):.3f}±{np.std(rmse_euclid):.3f} \
        MAE={np.mean(mae_euclid):.3f}±{np.std(mae_euclid):.3f} \
        R2={np.mean(r2_euclid):.3f}±{np.std(r2_euclid):.3f}")
print(f"Spherical RMSE={np.mean(rmse_spherical):.3f}±{np.std(rmse_spherical):.3f} \
        MAE={np.mean(mae_spherical):.3f}±{np.std(mae_spherical):.3f} \
        R2={np.mean(r2_spherical):.3f}±{np.std(r2_spherical):.3f}")

print(f"   \\val{{{np.mean(rmse_euclid):.3f}}}{{{np.std(rmse_euclid):.3f}}}"
      f" & \\val{{{np.mean(mae_euclid):.3f}}}{{{np.std(mae_euclid):.3f}}}"
      f" & \\val{{{np.mean(r2_euclid):.3f}}}{{{np.std(r2_euclid):.3f}}}")
print(f"   \\val{{{np.mean(rmse_spherical):.3f}}}{{{np.std(rmse_spherical):.3f}}}"
      f" & \\val{{{np.mean(mae_spherical):.3f}}}{{{np.std(mae_spherical):.3f}}}"
      f" & \\val{{{np.mean(r2_spherical):.3f}}}{{{np.std(r2_spherical):.3f}}}")


In [ ]:
"[☑️ SotA benchmarks]"

# ========================
"Conditional-VAE"

class CVAE:
    @staticmethod
    def run_cvae_scenario_LSTM(X_train, X_test, y_train, y_test, p: RunParams) -> Tuple[float, float, float]:
        X_train_s, X_test_s = scale_train_and_test_sets(X_train, X_test)
        X_train_w = Windowing.make_windows_from_X(X_train_s, p.window_size, sliding_size).to(device)
        X_test_w  = Windowing.make_windows_from_X(X_test_s, p.window_size, sliding_size).to(device)

        y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
        y_train_w = torch.tensor(Windowing.make_windows_from_y(y_train_s, p.window_size, sliding_size, task=p.task)).float().to(device)
        y_test_w  = torch.tensor(Windowing.make_windows_from_y(y_test_s, p.window_size, sliding_size, task=p.task)).float().to(device)

        y_dim = y_train_w.shape[-1]
        
        # LSTM Encoder: input_dim includes y_dim
        # Decoder (MLP): z_dim includes y_dim
        encoder      = LSTMEncoderEuclid(input_dim=X_train.shape[1] + y_dim, hidden_dim=p.hidden_dim, z_dim=p.z_dim_total).to(device)
        decoder      = MLPDecoder(z_dim_total=p.z_dim_total + y_dim, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=p.hidden_dim).to(device)
        optimizer    = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)
        train_loader = DataLoader(TensorDataset(X_train_w, y_train_w), batch_size=p.batch_size)

        for _ in range(p.epochs):
            for x, y_label in train_loader:
                optimizer.zero_grad()
                # For LSTM, we must repeat y across the time window: (N, y_dim) -> (N, Window, y_dim)
                y_repeated = y_label.unsqueeze(1).repeat(1, p.window_size, 1)
                x_cond     = torch.cat([x, y_repeated], dim=-1)
                
                mu, logvar = encoder(x_cond)
                z          = Reparam.reparam_gaussian(mu, logvar)
                
                # Decoder sees z + y
                z_cond = torch.cat([z, y_label], dim=-1)
                x_hat  = decoder(z_cond).view_as(x)
                
                loss = F.mse_loss(x_hat, x) + (p.lambda_kl_euc * kl_gaussian(mu, logvar))
                loss.backward()
                optimizer.step()

        encoder.eval()
        with torch.no_grad():
            # 1. Train set: Repeat y across time dimension
            y_train_rep = y_train_w.unsqueeze(1).repeat(1, p.window_size, 1) # (N, W, y_dim)
            Z_train, _  = encoder(torch.cat([X_train_w, y_train_rep], dim=-1))
            
            # 2. Test set (The Fair Test): Create 2D dummy, THEN repeat it
            y_dummy_2d  = torch.zeros_like(y_test_w) 
            y_dummy_rep = y_dummy_2d.unsqueeze(1).repeat(1, p.window_size, 1) # (N, W, y_dim)
            
            # Now both are 3D: (N, W, feat_dim) and (N, W, y_dim)
            Z_test, _   = encoder(torch.cat([X_test_w, y_dummy_rep], dim=-1))

        y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_w.cpu().numpy(), Z_test.cpu().numpy())
        rmse  = root_mean_squared_error(y_test_w.cpu().numpy(), y_hat)
        r2    = r2_score(y_test_w.cpu().numpy(), y_hat)
        mae   = mean_absolute_error(y_test_w.cpu().numpy(), y_hat)
        return rmse, r2, mae

    @staticmethod
    def run_cvae_scenario_MLP(X_train, X_test, y_train, y_test, p: RunParams) -> Tuple[float, float, float]:
        X_train_s, X_test_s = scale_train_and_test_sets(X_train, X_test)
        y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)

        # Windowing
        X_train_w = Windowing.make_windows_from_X(X_train_s, p.window_size, sliding_size).to(device)
        X_test_w  = Windowing.make_windows_from_X(X_test_s, p.window_size, sliding_size).to(device)
        y_train_w = torch.tensor(Windowing.make_windows_from_y(y_train_s, p.window_size, sliding_size, task=p.task)).float().to(device)
        y_test_w  = torch.tensor(Windowing.make_windows_from_y(y_test_s, p.window_size, sliding_size, task=p.task)).float().to(device)

        # CVAE Dimensions: Concatenate y to inputs
        y_dim = y_train_w.shape[-1]
        
        # Encoder sees [X, y] | Decoder sees [Z, y]
        encoder = EuclidEncoder(window_size=p.window_size, input_dim=X_train.shape[1] + y_dim, z_dim=p.z_dim_total, hidden=p.hidden_dim).to(device)
        decoder = Decoder(z_dim_total=p.z_dim_total + y_dim, window_size=p.window_size, output_dim=X_train.shape[1], hidden=p.hidden_dim).to(device)
        optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)

        train_loader = DataLoader(TensorDataset(X_train_w, y_train_w), batch_size=p.batch_size, shuffle=False)

        for _ in range(p.epochs):
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                # 1. Concat y to X for encoder
                # If MLP expects flat: batch_x is (N, W, D), flatten or repeat y to match
                # For simplicity, if tabular (W=1):
                x_cond = torch.cat([batch_x.squeeze(1), batch_y], dim=-1).unsqueeze(1)
                mu, logvar = encoder(x_cond)
                z = Reparam.reparam_gaussian(mu, logvar)
                
                # 2. Concat y to z for decoder
                z_cond = torch.cat([z, batch_y], dim=-1)
                x_hat = decoder(z_cond).view_as(batch_x)
                
                loss = F.mse_loss(x_hat, batch_x) + (p.lambda_kl_euc * kl_gaussian(mu, logvar))
                loss.backward()
                optimizer.step()

        # Encode for downstream: Pass X and dummy (zeros) or true y to get Z
        encoder.eval()
        with torch.no_grad():
            Z_train, _ = encoder(torch.cat([X_train_w.squeeze(1), y_train_w], dim=-1).unsqueeze(1))
            Z_test, _  = encoder(torch.cat([X_test_w.squeeze(1), y_test_w], dim=-1).unsqueeze(1))

        y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_w.cpu().numpy(), Z_test.cpu().numpy())
        rmse  = root_mean_squared_error(y_test_w.cpu().numpy(), y_hat)
        r2    = r2_score(y_test_w.cpu().numpy(), y_hat)
        mae   = mean_absolute_error(y_test_w.cpu().numpy(), y_hat)
        return rmse, r2, mae

# ==============
"beta-tcvae"

def log_density_gaussian(x, mu, logvar):
    """Calculates log p(x|mu, var) for the TC approximation."""
    normalization = -0.5 * (np.log(2 * np.pi) + logvar)
    inv_var = torch.exp(-logvar)
    log_density = normalization - 0.5 * ((x - mu)**2 * inv_var)
    return log_density

def calculate_total_correlation(z, mu, logvar):
    """
    Approximates Total Correlation: KL(q(z) || prod_j q(z_j)).
    This is the core of the beta-TCVAE baseline.
    """
    batch_size, z_dim = z.shape
    
    # log q(z|x)
    log_qz_prob = log_density_gaussian(z.view(batch_size, 1, z_dim), 
                                      mu.view(1, batch_size, z_dim), 
                                      logvar.view(1, batch_size, z_dim))
    
    # log q(z) approx
    log_qz = torch.logsumexp(log_qz_prob.sum(dim=2), dim=1) - np.log(batch_size)
    
    # log prod_j q(z_j) approx
    log_qz_product = (torch.logsumexp(log_qz_prob, dim=1) - np.log(batch_size)).sum(dim=1)
    
    return (log_qz - log_qz_product).mean()

def run_beta_tcvae_scenario(X_train, X_test, y_train, y_test, p: RunParams, beta=6.0):
    """
    Unified beta-TCVAE runner. 
    beta=1.0 makes this a standard VAE. 
    beta > 1.0 (usually 6.0 or 10.0) enables disentanglement.
    """
    # 1. Scaling & Windowing
    X_train_s, X_test_s = scale_train_and_test_sets(X_train, X_test)
    y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
    
    X_train_w = Windowing.make_windows_from_X(X_train_s, p.window_size, p.sliding_size).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test_s, p.window_size, p.sliding_size).to(device)
    y_train_win = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_s, p.window_size, p.sliding_size, task=p.task)

    # 2. Model Selection (MLP vs LSTM based on p.task)
    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
    
    if p.task == "tabular":
        encoder = EuclidEncoder(window_size=p.window_size, input_dim=X_train.shape[1], z_dim=p.z_dim_total, hidden=hidden_dim_split).to(device)
        decoder = Decoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden=hidden_dim_split).to(device)
    else:
        encoder = LSTMEncoderEuclid(input_dim=X_train.shape[1], hidden_dim=hidden_dim_split, z_dim=p.z_dim_total).to(device)
        decoder = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=hidden_dim_split).to(device)
    
    optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=True)

    # 3. Training Loop with TC Penalty
    encoder.train(); decoder.train()
    for _ in range(p.epochs):
        for (batch_x,) in train_loader:
            optimizer.zero_grad()
            
            mu, logvar = encoder(batch_x)
            z = Reparam.reparam_gaussian(mu, logvar)
            x_hat_flat = decoder(z)
            x_hat = x_hat_flat.view_as(batch_x)
            
            # Loss Components
            recon_loss = F.mse_loss(x_hat, batch_x)
            kl_loss    = kl_gaussian(mu, logvar) # Standard KL
            tc_loss    = calculate_total_correlation(z, mu, logvar) # Disentanglement Penalty
            
            # The beta-TCVAE objective: Recon + KL + (beta-1)*TC
            loss = p.lambda_recon * recon_loss + kl_loss + (beta - 1.0) * tc_loss
            
            loss.backward()
            optimizer.step()

    # 4. Evaluation (Encode + CatBoost)
    encoder.eval()
    with torch.no_grad():
        Z_train, _ = encoder(X_train_w)
        Z_test, _  = encoder(X_test_w)

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())
    
    rmse = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2   = r2_score(y_test_win, y_hat)
    mae  = mean_absolute_error(y_test_win, y_hat)
    return rmse, r2, mae

# ==============
"vamp"

class VAMPPrior(nn.Module):
    def __init__(self, K, input_dim, window_size, encoder, device):
        super(VAMPPrior, self).__init__()
        self.K = K
        self.encoder = encoder
        self.device = device
        # Learnable pseudo-inputs (u_k)
        self.pseudo_inputs = nn.Parameter(torch.randn(K, window_size, input_dim))
        self.idle_input = torch.eye(K, K).to(device) # For indexing

    def forward(self):
        # Pass pseudo-inputs through the encoder to get the mixture components
        mu, logvar = self.encoder(self.pseudo_inputs)
        return mu, logvar

def log_p_z_vamp(z, mu_p, logvar_p):
    """Calculates log p(z) under the VAMP mixture prior."""
    # z: (Batch, Z_dim) | mu_p: (K, Z_dim)
    z = z.unsqueeze(1)          # (Batch, 1, Z_dim)
    mu_p = mu_p.unsqueeze(0)    # (1, K, Z_dim)
    logvar_p = logvar_p.unsqueeze(0) # (1, K, Z_dim)

    # Log-density of each component
    log_q_z_u = log_density_gaussian(z, mu_p, logvar_p) # (Batch, K, Z_dim)
    
    # Sum over Z_dim, then LogSumExp over K components
    return torch.logsumexp(log_q_z_u.sum(dim=-1), dim=1) - np.log(mu_p.size(1))

def run_vamp_vae_scenario(X_train, X_test, y_train, y_test, p: RunParams, K=50):
    # 1. Standard Pre-processing
    X_train_s, X_test_s = scale_train_and_test_sets(X_train, X_test)
    y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
    
    X_train_w   = Windowing.make_windows_from_X(X_train_s, p.window_size, sliding_size).to(device)
    X_test_w    = Windowing.make_windows_from_X(X_test_s, p.window_size, sliding_size).to(device)
    y_train_win = Windowing.make_windows_from_y(y_train_s, p.window_size, sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_s, p.window_size, sliding_size, task=p.task)

    # 2. Model Init
    hidden_dim_split = int(p.hidden_dim / np.sqrt(2))
    if p.task == "tabular":
        encoder = EuclidEncoder(window_size=p.window_size, input_dim=X_train.shape[1], z_dim=p.z_dim_total, hidden=hidden_dim_split).to(device)
        decoder = Decoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden=hidden_dim_split).to(device)
    else:
        encoder = LSTMEncoderEuclid(input_dim=X_train.shape[1], hidden_dim=hidden_dim_split, z_dim=p.z_dim_total).to(device)
        decoder = MLPDecoder(z_dim_total=p.z_dim_total, window_size=p.window_size, output_dim=X_train.shape[1], hidden_dim=hidden_dim_split).to(device)

    # 1. Initialize models as before
    vamp_prior = VAMPPrior(K, X_train.shape[1], p.window_size, encoder, device).to(device)

    # 2. Use a set to filter out duplicate parameters
    all_params = list(encoder.parameters()) + list(decoder.parameters()) + list(vamp_prior.parameters())
    unique_params = list({id(p): p for p in all_params}.values())

    # 3. Pass unique_params to the optimizer
    optimizer = torch.optim.AdamW(unique_params, lr=p.lr_optimizer)

    # 3. Training
    for _ in range(p.epochs):
        for (batch_x,) in DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=True):
            optimizer.zero_grad()
            
            mu, logvar = encoder(batch_x)
            z = Reparam.reparam_gaussian(mu, logvar)
            x_hat = decoder(z).view_as(batch_x)
            
            # Reconstruction Loss
            recon_loss = F.mse_loss(x_hat, batch_x)
            
            # VAMP KL Divergence: E_q [log q(z|x) - log p(z)]
            log_qz = log_density_gaussian(z, mu, logvar).sum(dim=-1)
            mu_p, logvar_p = vamp_prior() # Get prior from pseudo-inputs
            log_pz = log_p_z_vamp(z, mu_p, logvar_p)
            
            kl_vamp = (log_qz - log_pz).mean()
            
            loss = p.lambda_recon * recon_loss + p.lambda_kl_euc * kl_vamp
            loss.backward()
            optimizer.step()

    # 4. Evaluation
    encoder.eval()
    with torch.no_grad():
        Z_train, _ = encoder(X_train_w)
        Z_test, _  = encoder(X_test_w)

    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())
    mae   = mean_absolute_error(y_test_win, y_hat)
    return np.sqrt(mean_squared_error(y_test_win, y_hat)), r2_score(y_test_win, y_hat), mae


rmse_cvae, r2_cvae, mae_cvae    = [], [], []
rmse_tcvae, r2_tcvae, mae_tcvae = [], [], []
rmse_vamp, r2_vamp, mae_vamp    = [], [], []
for i in range(5):
    rmse_cvae_i, r2_cvae_i, mae_cvae_i = CVAE.run_cvae_scenario_LSTM(X_train, X_test, y_train, y_test, p)
    rmse_cvae.append(rmse_cvae_i)
    r2_cvae.append(r2_cvae_i)
    mae_cvae.append(mae_cvae_i)

    rmse_tcvae_i, r2_tcvae_i, mae_tcvae_i = run_beta_tcvae_scenario(X_train, X_test, y_train, y_test, p, beta=4.0)
    rmse_tcvae.append(rmse_tcvae_i)
    r2_tcvae.append(r2_tcvae_i)
    mae_tcvae.append(mae_tcvae_i)

    rmse_vamp_i, r2_vamp_i, mae_vamp_i = run_vamp_vae_scenario(X_train, X_test, y_train, y_test, p, K=50)
    rmse_vamp.append(rmse_vamp_i)
    r2_vamp.append(r2_vamp_i)
    mae_vamp.append(mae_vamp_i)

print(f"C-VAE RMSE={np.mean(rmse_cvae):.3f}±{np.std(rmse_cvae):.3f} \
        MAE={np.mean(mae_cvae):.3f}±{np.std(mae_cvae):.3f} \
        R2={np.mean(r2_cvae):.3f}±{np.std(r2_cvae):.3f}")

print(f"beta-TC VAE RMSE={np.mean(rmse_tcvae):.3f}±{np.std(rmse_tcvae):.3f} \
        MAE={np.mean(mae_tcvae):.3f}±{np.std(mae_tcvae):.3f} \
        R2={np.mean(r2_tcvae):.3f}±{np.std(r2_tcvae):.3f}")

print(f"VAMP VAE RMSE={np.mean(rmse_vamp):.3f}±{np.std(rmse_vamp):.3f} \
        MAE={np.mean(mae_vamp):.3f}±{np.std(mae_vamp):.3f} \
        R2={np.mean(r2_vamp):.3f}±{np.std(r2_vamp):.3f}")


In [ ]:
"⚠️ torlin + beta-tcvae"

def log_density_mixed(z_combined, mu_e, logvar_e, mu_t, kappa_t):
    """Calculates log q(z|x) for TC approximation across hybrid geometries."""
    batch_size = z_combined.size(0)
    z_dim_e = mu_e.size(1)
    num_cyc = mu_t.size(1)

    # 1. Euclidean Density (Gaussian)
    z_e = z_combined[:, :z_dim_e].unsqueeze(1) 
    m_e = mu_e.unsqueeze(0)                    
    v_e = logvar_e.unsqueeze(0)                
    log_q_ze = -0.5 * (np.log(2 * np.pi) + v_e + (z_e - m_e)**2 * torch.exp(-v_e))
    
    # 2. Toroidal Density (von Mises approx)
    z_t = z_combined[:, z_dim_e:].view(batch_size, num_cyc, 2).unsqueeze(1) 
    m_t = mu_t.unsqueeze(0)                                                
    k_t = kappa_t.unsqueeze(0)                                             
    inner_prod = (z_t * m_t).sum(dim=-1) 
    log_norm_t = torch.log(2 * np.pi * torch.i0(k_t.squeeze(-1) + 1e-6))
    log_q_zt   = (k_t.squeeze(-1) * inner_prod) - log_norm_t
    return log_q_ze.sum(dim=-1) + log_q_zt.sum(dim=-1)

def calculate_mixed_tc(z_combined, mu_e, logvar_e, mu_t, kappa_t):
    """Approximates Total Correlation: KL(q(z) || prod_j q(z_j))."""
    batch_size     = z_combined.size(0)
    log_qz_prob    = log_density_mixed(z_combined, mu_e, logvar_e, mu_t, kappa_t)
    log_qz         = torch.logsumexp(log_qz_prob, dim=1) - np.log(batch_size)
    log_qz_product = torch.logsumexp(log_qz_prob, dim=1) - np.log(batch_size) # TC approximation
    return (log_qz - log_qz_product).mean()

def run_beta_tc_toroidal_scenario(X_train, X_test, y_train, y_test, p: RunParams, beta=6.0):
    # 1. Split & Preprocess
    X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose=False)
    X_lin_test, X_cyc_test   = X_test[X_lin_train.columns], X_test[X_cyc_train.columns]
    X_lin_train, X_lin_test  = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test  = scale_train_and_test_sets(X_cyc_train, X_cyc_test)
    
    # Windowing
    X_lin_tr_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size).to(device)
    X_cyc_tr_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size).to(device)
    X_lin_te_w = Windowing.make_windows_from_X(X_lin_test, p.window_size, p.sliding_size).to(device)
    X_cyc_te_w = Windowing.make_windows_from_X(X_cyc_test, p.window_size, p.sliding_size).to(device)
    
    num_cyc_features = X_cyc_train.shape[1]
    z_dim_spheric    = 2 * num_cyc_features
    z_dim_euclid     = max(8, p.z_dim_total - z_dim_spheric)
    
    # 2. Models
    hidden_split = int(p.hidden_dim / np.sqrt(2))
    enc_e        = LSTMEncoderEuclid(X_lin_train.shape[1], hidden_split, z_dim_euclid).to(device)
    enc_t        = LSTMToroidalEncoder(X_cyc_train.shape[1], hidden_split, num_cyc_features).to(device)
    decoder      = MLPDecoder(z_dim_total=z_dim_euclid + z_dim_spheric, window_size=p.window_size, 
                         output_dim=X_train.shape[1], hidden_dim=p.hidden_dim).to(device)
    optimizer = torch.optim.AdamW(list(enc_e.parameters()) + list(enc_t.parameters()) + list(decoder.parameters()), lr=p.lr_optimizer)
    loader    = DataLoader(TensorDataset(X_lin_tr_w, X_cyc_tr_w), batch_size=p.batch_size, shuffle=True)

    # 3. Training
    for epoch in range(p.epochs):
        enc_e.train(); enc_t.train(); decoder.train()
        for b_lin, b_cyc in loader:
            optimizer.zero_grad()
            mu_e, logvar_e = enc_e(b_lin)
            z_e            = reparam_gaussian(mu_e, logvar_e)
            mu_t, kappa_t  = enc_t(b_cyc)
            z_t            = torch.cat([sample_vmf(mu_t[:, i, :], kappa_t[:, i]) for i in range(num_cyc_features)], dim=-1)
            z_combined     = torch.cat([z_e, z_t], dim=-1)

            recon      = decoder(z_combined)
            target     = torch.cat([b_lin, b_cyc], dim=-1).view(b_lin.size(0), -1)
            loss_recon = F.mse_loss(recon.view(recon.size(0), -1), target)            
            tc_loss    = calculate_mixed_tc(z_combined, mu_e, logvar_e, mu_t, kappa_t)
            kl_euc     = -0.5 * torch.sum(1 + logvar_e - mu_e.pow(2) - logvar_e.exp(), dim=1).mean()
            kl_tor     = sum((kappa_t[:, i] - torch.log(kappa_t[:, i] + 1e-6)).mean() for i in range(num_cyc_features))
            loss       = p.lambda_recon * loss_recon + kl_euc + kl_tor + (beta - 1.0) * tc_loss
            loss.backward()
            optimizer.step()

    # 4. Evaluation
    enc_e.eval(); enc_t.eval()
    with torch.no_grad():
        def get_z_matrix(lin_win, cyc_win):
            me, _ = enc_e(lin_win)
            mt, _ = enc_t(cyc_win)
            zt    = torch.cat([F.normalize(mt[:, i, :], dim=-1) for i in range(num_cyc_features)], dim=-1)
            return torch.cat([me, zt], dim=-1).cpu().numpy()

        Z_train = get_z_matrix(X_lin_tr_w, X_cyc_tr_w)
        Z_test  = get_z_matrix(X_lin_te_w, X_cyc_te_w) 

    y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
    y_tr_win = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
    y_te_win = Windowing.make_windows_from_y(y_test_s, p.window_size, p.sliding_size, task=p.task)
    
    y_hat    = fit_catboost_multi(Z_train, y_tr_win, Z_test)
    mae      = mean_absolute_error(y_te_win, y_hat)
    return root_mean_squared_error(y_te_win, y_hat), r2_score(y_te_win, y_hat), mae

# 2. Run the Scenario
# beta=1.0 is a standard VAE; beta > 1.0 enables TC disentanglement
rmse_betatcvae_torlin, r2_betatcvae_torlin, mae_betatcvae_torlin = run_beta_tc_toroidal_scenario(X_train, X_test, y_train, y_test, p=p, 
                                                                           beta=6.0)
print(f"Beta-TCVAE+Torlin ({prediction_task}), RMSE={rmse_betatcvae_torlin:.4f}, MAE={mae_betatcvae_torlin:.4f}, R2={r2_betatcvae_torlin:.4f}")



In [ ]:
results = []

# Test combinations of Geometry (k) and Disentanglement (beta)
for k in [4, 6, 8]:
    for b in [1, 1.1, 2, 4.0, 6, 8, 10.0]:
        try:
            rmse, r2, _, _ = run_beta_tc_torlin_pipeline(
                X_lin_tr, X_lin_te, 
                X_cyc_tr, X_cyc_te, 
                y_train, y_test, 
                p=p, 
                beta=b, 
                k_circles=k)
            results.append({'k': k, 'beta': b, 'rmse': rmse, 'r2': r2})
            print(f"k_circles={k}, beta={b}: rmse: {rmse:.4f} R2: {r2:.4f}")
        except Exception as e:
            print(f"   >> Failed for k={k}, b={b}: {e}")

df_results = pd.DataFrame(results)
print(df_results.sort_values(by='rmse', ascending=True))


In [ ]:
"🔎 hyperparam search for sphlin/torlin"

results    = []
best_rmse  = float("inf")
best_params= None

num_trials = 50
for i in range(num_trials):
    p = RunParams(
        z_dim_total=random.choice(list(range(6, 40))),
        hidden_dim=random.choice(list(range(30, 80))),
        window_size= p.window_size, #random.choice([170]),
        lr_optimizer=10 ** random.uniform(-3, -2.5),
        batch_size=random.choice([256]),
        epochs=epochs,
        horizon=1,
        cyclic_threshold=0.3, #random.choice([0.2, 0.3, 0.59]),
        lambda_recon=random.uniform(0, 2.5),
        lambda_kl_euc=random.uniform(0.5, 2.5),
        lambda_kl_sph=random.uniform(0.5, 2.5),
        lambda_pred=random.uniform(0.6, 2.5),
        )

    # rmse, r2, mae, (Z_train, Z_test) = Torlin.run_sphetorlin_pipeline(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    #                                                                   p=p, angular_features_list=angular_col_names_list)
    # rmse, r2, mae, (_, _) = Torlin.run_torlin_pipeline(X_train, X_test, y_train, y_test, p, angular_col_names_list)

    print(f"lambda_recon={p.lambda_recon:.7f}, lambda_kl_euc={p.lambda_kl_euc:.7f}, lambda_kl_sph={p.lambda_kl_sph:.7f}, lambda_pred={p.lambda_pred:.7f}, params: {p}")
    rmse, r2, mae, _, _, _, _ = Sphlin.run_sphlin_LSTM(X_train, X_test, y_train, y_test, p, sliding_size=sliding_size, manually_set_cols=angular_col_names_list)
    results.append({**p.__dict__, "rmse": rmse, "r2": r2, "mae": mae})
    print(f"run #{i} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}, ")

    if rmse < best_rmse:
        best_rmse   = rmse
        best_params = p
        print(f"  👑 New best RMSE: {best_rmse:.4f} @ run #{i}")

results_df = pd.DataFrame(results).sort_values("rmse")
results_df.head()
results_df.to_csv("vae_param_sweep.csv", index=False, mode= 'a')


In [ ]:
"UMAP viz"
import plotly.express as px

def visualize_latent_umap_3d(Z, y_true, y_pred, title="3D UMAP: Product Manifold"):
    # 1. Scale and convert
    z_np = Z.detach().cpu().numpy() if torch.is_tensor(Z) else Z
    z_scaled = StandardScaler().fit_transform(z_np)
    
    # 2. Compute Error (Mean Absolute Error per sample)
    if y_true.ndim > 1:
        error = np.mean(np.abs(y_true - y_pred), axis=1)
    else:
        error = np.abs(y_true - y_pred)
    
    # 3. 3D UMAP Projection
    reducer = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.1, random_state=42)
    z_umap = reducer.fit_transform(z_scaled)
    
    # 4. Create DataFrame for Plotly
    df_plot = pd.DataFrame(z_umap, columns=['UMAP1', 'UMAP2', 'UMAP3'])
    df_plot['Error'] = error
    
    # 5. Interactive 3D Plot
    fig = px.scatter_3d(
        df_plot, x='UMAP1', y='UMAP2', z='UMAP3',
        color='Error',
        color_continuous_scale='Viridis',
        opacity=0.7,
        title=title,
        labels={'Error': 'MAE'})
    
    fig.update_traces(marker=dict(size=4))
    fig.show()

    return df_plot

def analyze_snake_clusters(df_plot):
    # Split the plot into high-error and low-error points
    high_error_threshold = df_plot['Error'].quantile(0.90) 
    print(f"90th Percentile MAE: {high_error_threshold:.2f}")
    
    # Check if high error is localized in one cluster
    # (Assuming UMAP1 splits your clusters)
    cluster_1 = df_plot[df_plot['UMAP1'] < df_plot['UMAP1'].mean()]
    cluster_2 = df_plot[df_plot['UMAP1'] >= df_plot['UMAP1'].mean()]
    
    print(f"Cluster 1 Mean Error: {cluster_1['Error'].mean():.2f}")
    print(f"Cluster 2 Mean Error: {cluster_2['Error'].mean():.2f}")

# 1. Run your scenario
rmse_our_method, r2_our_method, (Z_train, Z_test) = run_sphlin_LSTM(X_train, X_test, y_train, y_test, p)

# 2. Define the split point (Mirroring your function logic)
X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose=False)
X_lin_test = X_test[X_lin_train.columns]
X_cyc_test = X_test[X_cyc_train.columns]

cyc_ratio = X_cyc_train.shape[1] / X_train.shape[1] # Use your split ratio
z_dim_euclid = max(20, min(int(p.z_dim_total * (1 - cyc_ratio)), 28))

# 3. Slice the latent space
# Z_test is (N, z_dim_total). We split into [Linear | Cyclic]
Z_test_np = Z_test.cpu().numpy() if torch.is_tensor(Z_test) else Z_test
Z_linear = Z_test_np[:, :z_dim_euclid]
Z_cyclic = Z_test_np[:, z_dim_euclid:]

# 4. Get y_hat for the error coloring
y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)
y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
y_test_win  = Windowing.make_windows_from_y(y_test_scaled, p.window_size, p.sliding_size, task=p.task)

# y_hat = LinearRegression().fit(Z_train.cpu().numpy(), y_train_win).predict(Z_test.cpu().numpy())
y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())

# 5. Plot Separate Manifolds
# visualize_latent_umap_3d(Z_linear, y_test_win, y_hat, title="Euclidean Manifold (The Snake)")
# visualize_latent_umap_3d(Z_cyclic, y_test_win, X_cyc_test.iloc[:, 0].values, title="Spherical Z colored by Cyclic Feature")
# visualize_latent_umap_3d(Z_cyclic, y_test_win, y_hat, title="Spherical Manifold (The Ring)")

# 1. Align the feature to the windows
# We select the same indices that Windowing.make_windows_from_X uses
color_feature = X_cyc_test.iloc[p.window_size - 1 : : p.sliding_size, 0].values

# 2. Trim to match Z_cyclic exactly (handles potential rounding differences)
color_feature = color_feature[:Z_cyclic.shape[0]]

# 3. Plot - This should now show a smooth color gradient along the spring
visualize_latent_umap_3d(
    Z=Z_cyclic, 
    y_true=y_test_win, 
    y_pred=color_feature, # Using the feature as 'y_pred' to force it into the color slot
    title="The Product Manifold: Helical Structure")

# ========================
# --- Run ---
y_hat  = fit_catboost_multi(Z_train, y_train_w, Z_test)
df_res = visualize_latent_umap_3d(Z_test, y_test_w, y_hat)


# Simple split based on the UMAP x-axis
c1 = df_res[df_res['UMAP1'] < df_res['UMAP1'].median()]
c2 = df_res[df_res['UMAP1'] >= df_res['UMAP1'].median()]

print(f"Cluster 1 MAE: {c1['Error'].mean():.2f}")
print(f"Cluster 2 MAE: {c2['Error'].mean():.2f}")

X_cyc_test.iloc[:, 0].describe()


In [ ]:
"NEW split"
# ---------------------------
# 1. split linear vs cyclic features BEFORE scaling
# ---------------------------
X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=cyclic_threshold)
X_lin_test   = X_test[X_lin_train.columns]
X_cyc_test   = X_test[X_cyc_train.columns]

# ---------------------------
# 2. scale separately
# ---------------------------
y_scaler       = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled  = y_scaler.transform(y_test)

X_lin_scaler   = StandardScaler()
X_lin_train    = pd.DataFrame(X_lin_scaler.fit_transform(X_lin_train), columns=X_lin_train.columns)
X_lin_test     = pd.DataFrame(X_lin_scaler.transform(X_lin_test), columns=X_lin_train.columns)

X_cyc_scaler   = StandardScaler()
X_cyc_train    = pd.DataFrame(X_cyc_scaler.fit_transform(X_cyc_train), columns=X_cyc_train.columns)
X_cyc_test     = pd.DataFrame(X_cyc_scaler.transform(X_cyc_test), columns=X_cyc_train.columns)

X_lin_train  = torch.tensor(X_lin_train.values, dtype=torch.float32)
X_lin_test   = torch.tensor(X_lin_test.values, dtype=torch.float32)
X_cyc_train  = torch.tensor(X_cyc_train.values, dtype=torch.float32)
X_cyc_test   = torch.tensor(X_cyc_test.values, dtype=torch.float32)

# ---------------------------
# 3. latent dimensions split
# ---------------------------
z_dim_euclid  = z_dim_total * X_lin_train.shape[-1] // X_train.shape[-1]
z_dim_spheric = z_dim_total - z_dim_euclid  # ensure sum matches decoder input
print(f"Latent dims -> Euclidean: {z_dim_euclid}, Spherical: {z_dim_spheric}")

# ---------------------------
# 4. window the data
# ---------------------------
X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, window_size).to(device)
X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, window_size).to(device)
X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test, window_size).to(device)
X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test, window_size).to(device)
X_train_w     = torch.cat([X_lin_train_w, X_cyc_train_w], dim=-1).to(device)
X_test_w      = torch.cat([X_lin_test_w, X_cyc_test_w], dim=-1).to(device)

# ---------------------------
# 5. Shared Encoder / Decoder Setup
# input_dim for encoder is the TOTAL features (lin + cyc)
total_input_dim = X_train.shape[1] 
encoder_mixed = MixedEncoder(input_dim=total_input_dim * window_size, 
                             hidden_dim=hidden_dim_split, 
                             z_e_dim=z_dim_euclid, 
                             z_s_dim=z_dim_spheric).to(device)

decoder = Decoder(z_dim_total=z_dim_euclid + z_dim_spheric, 
                  window_size=window_size, 
                  output_dim=total_input_dim, 
                  hidden=hidden_dim).to(device)

optimizer = torch.optim.AdamW(list(encoder_mixed.parameters()) + list(decoder.parameters()), lr=lr_optimizer)

# 6. Modified Training Step for Specialization
def train_step_specialized(x_full_w, lin_idx, cyc_idx, encoder, decoder, lambdas):
    B, win, D = x_full_w.shape
    x_flat = x_full_w.view(B, -1)
    
    # Encoder head outputs
    mu_e, logvar_e, mu_s, kappa = encoder(x_flat)
    
    # Reparameterize
    z_e = Reparam.reparam_gaussian(mu_e, logvar_e)
    z_s = Reparam.reparam_vmf(mu_s, kappa)
    z = torch.cat([z_e, z_s], dim=-1)
    
    # Decode
    x_hat_flat = decoder(z)
    x_hat = x_hat_flat.view(B, win, D)
    
    # Specialization Loss: Gradients for z_e come from lin, z_s from cyc
    # We slice the original and reconstructed tensors
    loss_recon_lin = F.mse_loss(x_hat[:, :, lin_idx], x_full_w[:, :, lin_idx])
    loss_recon_cyc = F.mse_loss(x_hat[:, :, cyc_idx], x_full_w[:, :, cyc_idx])
    
    L_kl_e = kl_gaussian(mu_e, logvar_e)
    L_kl_s = kl_vmf_uniform(mu_s, kappa)
    
    return lambdas["reconstr"] * (loss_recon_lin + loss_recon_cyc) + \
           lambdas["euc"] * L_kl_e + lambdas["sph"] * L_kl_s

# 7. Training Loop
# Identify indices for slicing
lin_indices = torch.arange(X_lin_train.shape[1])
cyc_indices = torch.arange(X_lin_train.shape[1], total_input_dim)

train_loader = DataLoader(TensorDataset(X_train_w), batch_size=batch_size, shuffle=True)

time_start = time.time()
for epoch in range(epochs):
    encoder_mixed.train(); decoder.train()
    epoch_loss = 0
    for (batch_x,) in train_loader:
        optimizer.zero_grad()
        loss = train_step_specialized(batch_x, lin_indices, cyc_indices, encoder_mixed, decoder, lambdas)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {epoch_loss/len(train_loader):.4f}")
time_end = time.time()
print(f"Training time: {time_end - time_start:.2f}s")

# 8. Encoding & Latent Pooling
encoder_mixed.eval()
with torch.no_grad():
    # Process train
    mu_e, _, mu_s, _ = encoder_mixed(X_train_w.view(X_train_w.size(0), -1))
    Z_train = torch.cat([mu_e, mu_s], dim=-1).unsqueeze(1) # shape (N, 1, Z)
    # Since MixedEncoder doesn't naturally output window sequences here, 
    # we treat it as 1 window or use pool_latents if you modify forward to handle windows.
    Z_train = pool_latents(Z_train, z_e_dim=z_dim_euclid)

    # Process test
    mu_e_t, _, mu_s_t, _ = encoder_mixed(X_test_w.view(X_test_w.size(0), -1))
    Z_test = torch.cat([mu_e_t, mu_s_t], dim=-1).unsqueeze(1)
    Z_test = pool_latents(Z_test, z_e_dim=z_dim_euclid)

# 9. Regression (Keep same)
y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_scaled[:Z_train.shape[0]], Z_test.cpu().numpy())
rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
r2    = r2_score(y_test_win, y_hat)
print(f"Test RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
"""3. Gaussian Process Regression (GPR)"""
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

def run_gpr_on_latents(Z_train, y_train, Z_test, y_test):
    """
    Z_train: (N, latent_dim) - Concat of your Euclidean and Spherical latents
    y_train: (N, 1) - Scaled targets
    """
    # 1. Define the Kernel
    # RBF: The standard 'radial basis function' (smoothness)
    # WhiteKernel: Handles the noise in the VAE latents
    kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2)) + WhiteKernel(noise_level=1e-1)

    # 2. Initialize GPR
    # n_restarts_optimizer: GPR can get stuck in local optima; this runs it 10 times
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=20, random_state=42)

    # 3. Fit
    print("Fitting GPR (this may be slow for N > 5000)...")
    gpr.fit(Z_train, y_train)

    # 4. Predict (returns mean and standard deviation)
    y_hat, sigma_uncertainty = gpr.predict(Z_test, return_std=True)

    # 5. Evaluate
    rmse = np.sqrt(mean_squared_error(y_test, y_hat))
    r2 = r2_score(y_test, y_hat)
    
    return rmse, r2, sigma_uncertainty

Z_train_np = Z_train#.detach().cpu().numpy()
Z_test_np  = Z_test#.detach().cpu().numpy()

y_train_w = Windowing.make_windows_from_y(y_train, window_size, sliding_size, task=prediction_task)
y_test_w  = Windowing.make_windows_from_y(y_test, window_size, sliding_size, task=prediction_task)

# Ensure y is also NumPy (Windowing usually returns NumPy, but let's be safe)
y_train_np = y_train_w.cpu().numpy() if torch.is_tensor(y_train_w) else y_train_w
y_test_np  = y_test_w.cpu().numpy() if torch.is_tensor(y_test_w) else y_test_w

rmse_gpr, r2_gpr, uncertainty = run_gpr_on_latents(Z_train_np, y_train_np, Z_test_np, y_test_np)
print(f"RMSE: {rmse_gpr}, R2: {r2_gpr}")


In [ ]:
"WAE"
from geo_solvers import compute_rbf_kernel, wae_loss, train_wae, WAE

# --- Settings ---
layer_dims      = [hidden_dim]
latent_dim      = z_dim_total
n_clusters      = 13 #12
regressor_type  = "rf"     # "linear", "rf", "catboost"
beta_vae        = 1e-3
lambda_mmd_wae  = 10
input_dim       = X_train.shape[1]

def run_wae_scenario_windowed(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: np.ndarray,
                              y_test: np.ndarray, p) -> tuple[float, float, tuple[torch.Tensor, torch.Tensor]]:
    """Run WAE on time series with windowed X and y, then CatBoost regression on latents."""

    # ---- scale ----
    y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)
    X_train_scaled, X_test_scaled = scale_train_and_test_sets(X_train, X_test)

    # ---- windowing ----
    X_train_w = Windowing.make_windows_from_X(X_train_scaled, p.window_size, p.sliding_size).to(device)
    X_test_w  = Windowing.make_windows_from_X(X_test_scaled,  p.window_size, p.sliding_size).to(device)

    # ---- create DataLoader ----
    train_loader = DataLoader(TensorDataset(X_train_w), batch_size=p.batch_size, shuffle=True)

    # ---- initialize WAE ----
    wae_model = WAE(input_dim=X_train_w.shape[-1], latent_dim=p.z_dim_total, layer_dims=[p.hidden_dim]).to(device)

    # ---- train WAE ----
    train_wae(wae_model, train_loader, epochs=p.epochs, lr=p.lr_optimizer, lambda_mmd=p.lambda_latent, device=device)

    # ---- encode latents ----
    with torch.no_grad():
        Z_train = wae_model.encode(X_train_w)
        Z_test  = wae_model.encode(X_test_w)

    # ---- window y to match latents ----
    y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, task=p.task)
    y_test_win  = Windowing.make_windows_from_y(y_test_scaled,  p.window_size, p.sliding_size, task=p.task)

    # ---- regression ----
    y_hat = fit_catboost_multi(Z_train.cpu().numpy(), y_train_win, Z_test.cpu().numpy())
    rmse  = np.sqrt(mean_squared_error(y_test_win, y_hat))
    r2    = r2_score(y_test_win, y_hat)

    return rmse, r2, (Z_train, Z_test)

rmse, r2, (Z_train, Z_test) = run_wae_scenario_windowed(X_train, X_test, y_train, y_test, p)
print(f"WAE + CatBoost RMSE: {rmse:.4f}, R2: {r2:.4f}")


In [ ]:
"tabular dataset processing"
z_lin_dim, z_s_dim   = 12, 3
hidden_lin, hidden_s = 32, 16
# cyc_cols         = ["Latitude", "Longitude"] # california
# cyc_cols = ['mnth', 'hr', 'weekday'] # bike sharing
cyc_cols = ['month', 'day'] # forest fires


def run_tabular_latent(X_train: pd.DataFrame, X_test: pd.DataFrame, 
                       y_train: np.ndarray, y_test: np.ndarray, 
                       cyc_features: list,
                       encoding: str = "cartesian",
                       z_lin_dim: int = 18, z_s_dim: int = 4,
                       hidden_lin: int = 40, hidden_s: int = 25):
    """Encode tabular data into latent z and predict y. Works with any cyclic/linear split."""

    # --- Identify linear features ---
    lin_features = [f for f in X_train.columns if f not in cyc_features]

    X_lin_train, X_cyc_train = X_train[lin_features].values, X_train[cyc_features].values
    X_lin_test,  X_cyc_test  = X_test[lin_features].values,  X_test[cyc_features].values

    # --- Scale ---
    y_train, y_test         = scale_train_and_test_sets(y_train, y_test)
    X_lin_train, X_lin_test = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

    # --- Convert to tensors on device ---
    X_lin_train = torch.as_tensor(X_lin_train, dtype=torch.float32, device=device)
    X_lin_test  = torch.as_tensor(X_lin_test,  dtype=torch.float32, device=device)
    X_cyc_train = torch.as_tensor(X_cyc_train, dtype=torch.float32, device=device)
    X_cyc_test  = torch.as_tensor(X_cyc_test,  dtype=torch.float32, device=device)

    # --- Define encoders ---
    lin_encoder = EuclidEncoder(1, X_lin_train.shape[1], z_lin_dim, hidden_lin).to(device)
    cyc_encoder = SphericalEncoder(1, X_cyc_train.shape[1], z_s_dim, hidden_s).to(device)

    # --- Forward pass ---
    if encoding == "cartesian":
        z_s_train   = torch.cat([X_cyc_train, torch.zeros(X_cyc_train.shape[0], 1, device=device)], dim=1)
        z_s_test    = torch.cat([X_cyc_test,  torch.zeros(X_cyc_test.shape[0], 1, device=device)], dim=1)
        z_lin_train = lin_encoder(X_lin_train)[0]
        z_lin_test  = lin_encoder(X_lin_test)[0]
    elif encoding == "vmf":
        mu_s_train, kappa_train = cyc_encoder(X_cyc_train)
        z_s_train = Reparam.sample_vmf(mu_s_train, kappa_train)
        mu_s_test, kappa_test   = cyc_encoder(X_cyc_test)
        z_s_test  = Reparam.sample_vmf(mu_s_test, kappa_test)
        z_lin_train, _ = lin_encoder(X_lin_train)
        z_lin_test, _  = lin_encoder(X_lin_test)
    elif encoding == "gaussian":
        z_lin_train, _ = lin_encoder(X_lin_train)
        z_lin_test, _  = lin_encoder(X_lin_test)
        z_s_train = cyc_encoder(X_cyc_train)[0]
        z_s_test  = cyc_encoder(X_cyc_test)[0]
    else:
        raise ValueError("encoding must be cartesian / vmf / gaussian")

    # --- Combine latents ---
    Z_train = torch.cat([z_lin_train, z_s_train], dim=1).detach().cpu().numpy()
    Z_test  = torch.cat([z_lin_test,  z_s_test],  dim=1).detach().cpu().numpy()

    # --- Predict ---
    y_hat = fit_catboost_multi(Z_train, y_train, Z_test)
    rmse  = np.sqrt(mean_squared_error(y_test, y_hat))
    r2    = r2_score(y_test, y_hat)
    return rmse, r2


for enc in ["cartesian", "vmf", "gaussian"]:
    rmse, r2 = run_tabular_latent(X_train, X_test, y_train, y_test, cyc_features=cyc_cols, encoding=enc)
    print(f"{enc:8s} latent: RMSE={rmse:.4f}, R2={r2:.4f}")


In [ ]:
"Ts2vec"
from src.ts2vec_model.ts2vec import TS2Vec # the real ts2vec library
from src.encoders.ts2vec_encoder import TS2VecEncoder
from src.utils.metrics_utils import Preds

# below uses the original ts2vec (class from see ts2vec_encoder.py)
class TS2VecEncoder:
    """TS2Vec + Linear Predictor pipeline with externally set hyperparameters
       Supports early stopping during TS2Vec training."""

    def __init__(self, lr, z_pooling: str = "mean", device=None, patience: int = 20, max_train_length: int = 512):
        """Args:
            z_pooling (str): Pooling method for encoding.
            device (str or torch.device): "cpu" or "cuda".
            patience (int): Number of epochs with no improvement before stopping"""
        self.z_pooling   = z_pooling
        self.ts_model    = None
        self.device      = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.patience    = patience
        self._stop_early = False  # flag to indicate if early stopping occurred
        self.lr          = lr
        self.max_train_length = max_train_length

    class EarlyStopper:
        """Callback for TS2Vec early stopping."""
        def __init__(self, patience: int):
            self.patience = patience
            self.best = float("inf")
            self.wait = 0
            self.stop = False

        def __call__(self, model, loss: float):
            if loss < self.best:
                self.best = loss
                self.wait = 0
            else:
                self.wait += 1
                if self.wait >= self.patience:
                    print(f"Early stopping at epoch {model.n_epochs}")
                    self.stop = True
                    model.n_epochs = 1e9  # forces loop exit

    def fit_ts2vec(self, X_train, hidden_dims=12, output_dims=8, depth=5, batch_size=16, n_epochs=20):
        """Fit TS2Vec on training data with early stopping."""
        stopper       = self.EarlyStopper(self.patience)
        self.ts_model = TS2Vec(input_dims=X_train.shape[2], hidden_dims=hidden_dims, depth=depth, lr=self.lr,
                               output_dims=output_dims, batch_size=batch_size, device=self.device,
                               after_epoch_callback=stopper, max_train_length=self.max_train_length, temporal_unit=1)
        # self.ts_model.fit(X_train.astype(np.float32), n_epochs=n_epochs, verbose=True)
        X_np = X_train.cpu().numpy() if torch.is_tensor(X_train) else X_train
        self.ts_model.fit(X_np.astype(np.float32), n_epochs=n_epochs, verbose=True)


    def encode(self, X, pooling: str | None = "mean") -> np.ndarray:
        """Encode input X using trained TS2Vec model.
        Args:
            X: np.ndarray of shape (N, T, D)
            pooling: "mean" or None
        Returns:
            z: np.ndarray of shape (N, latent_dim) if pooled, else (N, T, latent_dim)"""
        if self.ts_model is None:
            raise ValueError("TS2Vec model not trained yet. Call fit_ts2vec first.")

        X_np = X.astype(np.float32) if isinstance(X, np.ndarray) else X.cpu().numpy()
        z    = self.ts_model.encode(X_np)  # (N, T, latent_dim)

        if pooling == "mean":
            return z.mean(axis=1)  # pool over time
        if pooling is None:
            return z
        raise ValueError(f"Unknown pooling: {pooling}")

# from ts2vec_runner.py, runs TS2VecEncoder class 
def run_ts2vec(X_train, X_test, y_train_scaled, y_test_scaled, *,
               model_cfg: dict, train_cfg: dict, device, save_ts2vec_encoder=False,):
    """Train TS2Vec with config dicts. Returns (losses, profiling_metrics)."""

    ts2vec = TS2VecEncoder(
        z_pooling= model_cfg["z_pooling_method"],
        lr       = train_cfg["lr"],
        device   = device,
        patience = train_cfg["patience"],
        max_train_length=train_cfg["window_size"],)

    metrics = ts2vec.fit_ts2vec(X_train,
        hidden_dims= model_cfg["hidden_dims"],
        output_dims= model_cfg["latent_dims"],
        depth      = model_cfg["depth"],
        batch_size = train_cfg["batch_size"],
        n_epochs   = train_cfg["epochs"],)

    z_train = ts2vec.encode(X_train, pooling=None)
    z_test  = ts2vec.encode(X_test,  pooling=None)

    print(f"{z_train.shape=}, {z_test.shape=}")

    z_train_flat = z_train.reshape(len(z_train), -1)
    z_test_flat  = z_test.reshape(len(z_test),  -1)

    print(f"{z_train_flat.shape=}, {z_test_flat.shape=}")

    losses, rf_model= Preds().evaluate_models_on_dataset(z_train_flat, y_train_scaled, z_test_flat,  y_test_scaled)
    r2              = rf_model.score(z_test_flat, y_test_scaled)
    mae             = mean_absolute_error(y_test_scaled, rf_model.predict(z_test_flat))
    return losses, r2, mae, metrics

# --- configs ---
model_cfg = {
    "z_pooling_method": "mean",  # how TS2Vec pools over time
    "hidden_dims": p.hidden_dim,
    "latent_dims": p.z_dim_total,
    "depth": 5,}

train_cfg = {
    "lr": 1e-3,
    "patience": 20,
    "window_size": 128,  # max sequence length for TS2Vec
    "batch_size": 16,
    "epochs": 20,}

# --- scale ---
X_train_t, X_test_t           = scale_train_and_test_sets(X_train, X_test)
y_train_scaled, y_test_scaled = scale_train_and_test_sets(y_train, y_test)
X_train_np = X_train_t.cpu().numpy() if torch.is_tensor(X_train_t) else np.array(X_train_t, dtype=np.float32)
X_test_np  = X_test_t.cpu().numpy()  if torch.is_tensor(X_test_t)  else np.array(X_test_t,  dtype=np.float32)

# --- make sliding windows ---
X_train_win = Windowing.make_windows_from_X(X_train_np, p.window_size, p.sliding_size)  # (num_windows, window_size, D)
X_test_win  = Windowing.make_windows_from_X(X_test_np,  p.window_size, p.sliding_size)
y_train_win = Windowing.make_windows_from_y(y_train_scaled, p.window_size, p.sliding_size, prediction_task)
y_test_win  = Windowing.make_windows_from_y(y_test_scaled,  p.window_size, p.sliding_size, prediction_task)

print(f"X_train_win: {X_train_win.shape}, y_train_win: {y_train_win.shape}")
print(f"X_test_win:  {X_test_win.shape}, y_test_win:  {y_test_win.shape}")

# --- run TS2Vec ---
losses, r2, mae, metrics = run_ts2vec(X_train_win, X_test_win, y_train_win, y_test_win,
                                      model_cfg=model_cfg, train_cfg=train_cfg, device=device)
print(f"Simple TS2Vec: Losses={losses}, MAE={mae:.4f}, R2={r2:.4f}")

# --- encode to DataFrame safely ---
z_train_flat, z_test_flat = (np.array(X_train_win).reshape(len(X_train_win), -1), np.array(X_test_win).reshape(len(X_test_win), -1))
Z_train_df = pd.DataFrame(z_train_flat)
Z_test_df  = pd.DataFrame(z_test_flat)

# --- feed into hybrid split-VAE ---
# rmse, r2_hybrid = Sphlin.run_sphlin_LSTM(Z_train_df, Z_test_df, y_train_win, y_test_win, p)
rmse_hybrid, r2_hybrid, mae_hybrid, (Z_train, Z_test), (y_hat, y_tr_w, y_te_w), (z_euc, z_sph) = Sphlin.run_sphlin_LSTM(X_train, X_test, y_train, y_test, p)
print(f"Hybrid VAE after TS2Vec: RMSE={rmse_hybrid:.4f}, R2={r2_hybrid:.4f}")



In [ ]:
"ts2vec part 2"

def run_ts2vec_plain(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: np.ndarray,
    y_test: np.ndarray,
    p: RunParams,
    model_cfg: dict,
    train_cfg: dict,
    device):
    """Baseline: full X → TS2Vec → regress (no geometry)"""

    # -------------------------
    # 0. Scale
    # -------------------------
    y_train_s, y_test_s = scale_train_and_test_sets(y_train, y_test)
    X_train, X_test     = scale_train_and_test_sets(X_train, X_test)

    # -------------------------
    # 1. Window X and y
    # -------------------------
    X_train_w = Windowing.make_windows_from_X(X_train, p.window_size, p.sliding_size)
    X_test_w  = Windowing.make_windows_from_X(X_test,  p.window_size, p.sliding_size)

    y_train_w = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
    y_test_w  = Windowing.make_windows_from_y(y_test_s,  p.window_size, p.sliding_size, task=p.task)

    # -------------------------
    # 2. TS2Vec encode
    # -------------------------
    ts = TS2VecEncoder(lr=train_cfg["lr"], device=device, patience=train_cfg["patience"])
    ts.fit_ts2vec(
        X_train_w.numpy(),
        hidden_dims=model_cfg["hidden_dims"],
        output_dims=model_cfg["latent_dims"],
        depth=model_cfg["depth"],
        batch_size=train_cfg["batch_size"],
        n_epochs=train_cfg["epochs"],
    )

    z_train = ts.encode(X_train_w.numpy(), pooling="mean")
    z_test  = ts.encode(X_test_w.numpy(),  pooling="mean")

    assert z_train.shape[0] == y_train_w.shape[0]
    assert z_test.shape[0]  == y_test_w.shape[0]

    # -------------------------
    # 3. Regress
    # -------------------------
    y_hat = fit_catboost_multi(z_train, y_train_w, z_test)
    rmse  = np.sqrt(mean_squared_error(y_test_w, y_hat))
    r2    = r2_score(y_test_w, y_hat)

    return rmse, r2


def run_split_ts2vec_hybrid(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: np.ndarray,
    y_test: np.ndarray,
    p: RunParams,
    model_cfg: dict,
    train_cfg: dict,
    device):
    """Option 2: split → TS2Vec separately → spherical projection → regress"""

    # -------------------------
    # 0. Split + scale
    # -------------------------
    X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train, threshold=p.cyclic_threshold, verbose=False)
    X_lin_test  = X_test[X_lin_train.columns]
    X_cyc_test  = X_test[X_cyc_train.columns]

    y_train_s, y_test_s     = scale_train_and_test_sets(y_train, y_test)
    X_lin_train, X_lin_test = scale_train_and_test_sets(X_lin_train, X_lin_test)
    X_cyc_train, X_cyc_test = scale_train_and_test_sets(X_cyc_train, X_cyc_test)

    # -------------------------
    # 1. Window X
    # -------------------------
    X_lin_train_w = Windowing.make_windows_from_X(X_lin_train, p.window_size, p.sliding_size)
    X_cyc_train_w = Windowing.make_windows_from_X(X_cyc_train, p.window_size, p.sliding_size)
    X_lin_test_w  = Windowing.make_windows_from_X(X_lin_test,  p.window_size, p.sliding_size)
    X_cyc_test_w  = Windowing.make_windows_from_X(X_cyc_test,  p.window_size, p.sliding_size)

    # -------------------------
    # 2. Window y  ✅ CRITICAL
    # -------------------------
    y_train_w = Windowing.make_windows_from_y(y_train_s, p.window_size, p.sliding_size, task=p.task)
    y_test_w  = Windowing.make_windows_from_y(y_test_s,  p.window_size, p.sliding_size, task=p.task)

    # -------------------------
    # 3. TS2Vec encoders
    # -------------------------
    ts_lin = TS2VecEncoder(lr=train_cfg["lr"], device=device, patience=train_cfg["patience"])
    ts_lin.fit_ts2vec(X_lin_train_w.numpy(), hidden_dims=model_cfg["hidden_dims"],
                      output_dims=model_cfg["latent_dims"], depth=model_cfg["depth"],
                      batch_size=train_cfg["batch_size"], n_epochs=train_cfg["epochs"])

    ts_cyc = TS2VecEncoder(lr=train_cfg["lr"], device=device, patience=train_cfg["patience"])
    ts_cyc.fit_ts2vec(X_cyc_train_w.numpy(), hidden_dims=model_cfg["hidden_dims"],
                      output_dims=model_cfg["latent_dims"], depth=model_cfg["depth"],
                      batch_size=train_cfg["batch_size"], n_epochs=train_cfg["epochs"])

    z_lin_train = ts_lin.encode(X_lin_train_w.numpy(), pooling="mean")
    z_lin_test  = ts_lin.encode(X_lin_test_w.numpy(),  pooling="mean")
    z_cyc_train = ts_cyc.encode(X_cyc_train_w.numpy(), pooling="mean")
    z_cyc_test  = ts_cyc.encode(X_cyc_test_w.numpy(),  pooling="mean")

    # -------------------------
    # 4. Spherical projection (cyclic only)
    # -------------------------
    z_cyc_train /= np.linalg.norm(z_cyc_train, axis=1, keepdims=True)
    z_cyc_test  /= np.linalg.norm(z_cyc_test,  axis=1, keepdims=True)

    # -------------------------
    # 5. Combine + regress
    # -------------------------
    Z_train = np.concatenate([z_lin_train, z_cyc_train], axis=1)
    Z_test  = np.concatenate([z_lin_test,  z_cyc_test],  axis=1)

    assert Z_train.shape[0] == y_train_w.shape[0]
    assert Z_test.shape[0]  == y_test_w.shape[0]

    y_hat = fit_catboost_multi(Z_train, y_train_w, Z_test)
    rmse  = np.sqrt(mean_squared_error(y_test_w, y_hat))
    r2    = r2_score(y_test_w, y_hat)

    return rmse, r2

# CONFIG
model_cfg = {
    "hidden_dims": 64,
    "latent_dims": 16,
    "depth": 5,}

train_cfg = {
    "lr": 1e-3,
    "patience": 20,
    "batch_size": 16,
    "epochs": 2}, #epoch,

# RUN
rmse_plain, r2_plain = run_ts2vec_plain(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    p=p,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device=device,)

rmse_split, r2_split = run_split_ts2vec_hybrid(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    p=p,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device=device,)

print(f"TS2Vec plain        : RMSE={rmse_plain:.4f}, R2={r2_plain:.4f}")
print(f"Split + geometry    : RMSE={rmse_split:.4f}, R2={r2_split:.4f}")


In [ ]:
"dimensionality estimation"
import skdim
from sklearn.decomposition import PCA
import geomstats.geometry.hypersphere as hs
from geomstats.learning.pca import TangentPCA

# 1. Prepare Data
X_array = X.values.astype(np.float32)
X_flattened = X_array 
n_samples, ambient_dim = X_flattened.shape

# --- Intrinsic Dimension Estimation ---

# A. TwoNN & B. MLE (Geometric/Local)
# These are often the most reliable for manifold learning
d_twonn = skdim.id.TwoNN().fit_transform(X_flattened)
d_mle   = skdim.id.MLE().fit_transform(X_flattened)

# C. PCA (Global Linear) - 90% Variance threshold
pca = PCA().fit(X_flattened)
id_pca = np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9) + 1

# D. PGA (Spherical Manifold)
# Hypersphere dim is (ambient - 1)
sphere = hs.Hypersphere(dim=ambient_dim - 1)
norms  = np.linalg.norm(X_flattened, axis=1, keepdims=True)
X_proj = X_flattened / np.where(norms == 0, 1.0, norms)

# Limit n_components to avoid memory issues in high-D
pga = TangentPCA(sphere, n_components=min(n_samples, ambient_dim, 100)) 
pga.fit(X_proj)
id_pga = np.searchsorted(np.cumsum(pga.explained_variance_ratio_), 0.9) + 1

# --- Final Decisions ---
print(f"TwoNN ID: {d_twonn:.2f}")
print(f"MLE ID:   {d_mle:.2f}")
print(f"PCA ID:   {id_pca}")
print(f"PGA ID:   {id_pga}")

# Decision Rule: 
# Usually, d_twonn or d_mle represent the 'true' manifold dimension.
# PCA/PGA represent the linear 'embedding' dimension.
final_z_dim = int(np.ceil(max(d_twonn, d_mle, id_pca, id_pga)))

print(f"\nRecommended VAE z_dim: {final_z_dim}")

In [ ]:
"kernel PCA"
from sklearn.decomposition import KernelPCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

# ===== 1. Sweep embedding dimension to estimate residual variance (optional) =====
# Note: KernelPCA does not give a built-in dist_matrix_, so residual variance calculation is less straightforward.
# Here we can skip it or just look at explained variance if using linear kernel.
# For nonlinear kernels, you typically pick n_components based on domain knowledge or grid search.

# ===== 2. Fit kernel PCA with chosen embedding dimension =====
n_components = 8  # same as your Isomap example
kpca_model   = KernelPCA(n_components=n_components, kernel='rbf', gamma=0.05, fit_inverse_transform=True)
X_train_kpca = kpca_model.fit_transform(X_train)
X_test_kpca  = kpca_model.transform(X_test)

# ===== 3. Plot first 3 components (for visualization) =====
fig = plt.figure(figsize=(8,6))
ax  = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_kpca[:,0],
    X_train_kpca[:,1],
    X_train_kpca[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')
ax.set_title("Kernel PCA 3D Embedding")
plt.show()

# ===== 4. RandomForestClassifier on KPCA embedding =====
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_kpca, y_train_np)
y_pred_class = rf_classifier.predict(X_test_kpca)
accuracy     = accuracy_score(y_test_np, y_pred_class)
rmse_class   = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Kernel PCA accuracy (classif.): {accuracy:.3f}")
print(f"Kernel PCA RMSE (classif.): {rmse_class:.3f}")

# ===== 5. RandomForestRegressor on KPCA embedding =====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_kpca, y_train_np)
y_pred_reg = rf_regressor.predict(X_test_kpca)
r2        = r2_score(y_test_np, y_pred_reg)
rmse_reg  = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Kernel PCA R² (regression): {r2:.3f}")
print(f"Kernel PCA RMSE (regression): {rmse_reg:.3f}")


In [ ]:
"isomap"
from sklearn.manifold import Isomap
import warnings
from scipy.sparse import SparseEfficiencyWarning
warnings.simplefilter('ignore', SparseEfficiencyWarning)

# ===== 1. plot residual variance to estimate intrinsic dimensionality (finds the best dim)
res_vars       = []
embedding_dims = range(1, 11)  # try 1D to 10D embeddings
for dim in embedding_dims:
    iso = Isomap(n_neighbors=20, n_components=dim)
    iso.fit(X_train)
    # residual variance = 1 - R^2 between graph distances and embedding distances
    dist_graph = iso.dist_matrix_
    dist_emb   = np.linalg.norm(iso.embedding_[:, None, :] - iso.embedding_[None, :, :], axis=2)
    r2         = np.corrcoef(dist_graph.ravel(), dist_emb.ravel())[0,1]**2
    res_vars.append(1 - r2)
plt.plot(embedding_dims, res_vars, marker='o')
plt.xlabel("Embedding dimension")
plt.ylabel("Residual variance")
plt.title("Estimate intrinsic dimensionality")
plt.show()

# ===== 2. plot isomap of training data (uses the best estimated dim from above)
isomap_model   = Isomap(n_neighbors=20, n_components=5)
X_train_isomap = isomap_model.fit_transform(X_train)
X_test_isomap  = isomap_model.transform(X_test)

fig     = plt.figure(figsize=(8,6))
ax      = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_isomap[:,0],
    X_train_isomap[:,1],
    X_train_isomap[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')  # attach to figure
ax.set_title("Isomap 3D Embedding")
plt.show()


In [ ]:
"isomap results"
rf_classifier= RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_isomap, y_train_np)          # train on Isomap embedding
y_pred_class = rf_classifier.predict(X_test_isomap)          # predict test labels
accuracy     = accuracy_score(y_test_np, y_pred_class)
print(f"Isomap accuracy (classif.): {accuracy:.3f}")
rmse         = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Isomap RMSE (classif.): {rmse:.3f}")

# ====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_isomap, y_train_np)           # train on Isomap embedding
y_pred_reg   = rf_regressor.predict(X_test_isomap)      # predict on test embedding
r2           = r2_score(y_test_np, y_pred_reg)
print(f"Isomap R² (regression): {r2:.3f}")
test_rmse    = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Isomap RMSE (regression): {test_rmse:.3f}")


In [ ]:
"new metrics"
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score

def propagate_knn(Z_latent, y_true, missing_frac=0.2, k=15, tau=1.0):
    """Perform kNN label propagation on latent vectors."""
    num_rows = Z_latent.shape[0]
    dist_matrix = np.zeros((num_rows, num_rows))
    
    # compute L2 distances
    for i in range(num_rows):
        u = Z_latent[i]
        for j in range(i, num_rows):
            v = Z_latent[j]
            d = np.linalg.norm(u - v)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # mask random fraction
    num_missing = int(missing_frac * num_rows)
    missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)
    y_input     = y_true.copy().astype(float)
    y_input[missing_idx] = np.nan

    # kNN neighbors
    neighbors_idx = np.argsort(dist_matrix, axis=1)[:, 1:k+1]

    # edge weights
    def compute_weights(dists):
        w = np.exp(-dists / tau)
        return w / w.sum()

    # propagate labels
    y_prop = y_input.copy()
    for i in range(num_rows):
        if np.isnan(y_input[i]):
            neigh     = neighbors_idx[i]
            neigh_y   = y_input[neigh]
            mask      = ~np.isnan(neigh_y)
            if np.sum(mask) == 0:
                continue
            neigh_y   = neigh_y[mask]
            neigh_d   = dist_matrix[i, neigh][mask]
            w         = compute_weights(neigh_d)
            y_prop[i] = np.sum(w * neigh_y)

    return y_prop, missing_idx

def evaluate_metrics(y_true, y_pred, missing_idx):
    """Compute RMSE, MAE, Accuracy, Spearman correlation, and R² for masked points."""
    y_true_masked = y_true[missing_idx]
    y_pred_masked = y_pred[missing_idx]
    mask_valid    = ~np.isnan(y_pred_masked)
    
    if mask_valid.sum() == 0:
        return {k: np.nan for k in ["rmse","mae","acc","spearman","r2"]}
    
    y_true_valid = y_true_masked[mask_valid]
    y_pred_valid = y_pred_masked[mask_valid]


    def safe_spearmanr(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y).correlation

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred_valid)),
        "mae": mean_absolute_error(y_true_valid, y_pred_valid),
        "acc": accuracy_score(y_true_valid, np.round(y_pred_valid).astype(int)),
        "spearman": safe_spearmanr(y_true_valid, y_pred_valid),
        "r2": r2_score(y_true_valid, y_pred_valid)
    }
    return metrics

def format_metrics(metrics: dict) -> dict:
    """Format all float metrics to 3 decimal places, keep nan as-is."""
    return {k: (f"{v:.3f}" if isinstance(v, float) and not np.isnan(v) else v)
            for k, v in metrics.items()}

# --- Hybrid AE ---
y_prop_hybrid, missing_idx = propagate_knn(z_hybrid_encoded_np, df.iloc[:,1].values)
metrics_hybrid = evaluate_metrics(df.iloc[:,1].values, y_prop_hybrid, missing_idx)
print("Hybrid metrics:", format_metrics(metrics_hybrid))

# --- Full Euclidean AE ---
y_prop_euc, missing_idx = propagate_knn(z_euclid_full_np, df.iloc[:,1].values)
metrics_euc = evaluate_metrics(df.iloc[:,1].values, y_prop_euc, missing_idx)
print("Full Euclidean metrics:", format_metrics(metrics_euc))

# --- Baseline (mean predictor) ---
y_true      = df.iloc[:,1].values
num_rows    = len(y_true)
num_missing = int(0.2 * num_rows)
np.random.seed(42)
missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)

y_baseline = y_true.copy().astype(float)
y_baseline[missing_idx] = np.nan
mean_label = np.nanmean(y_baseline)
y_baseline[np.isnan(y_baseline)] = mean_label

metrics_baseline = evaluate_metrics(y_true, y_baseline, missing_idx)
print("Baseline metrics:", format_metrics(metrics_baseline))
